# FD calibration — Tap300_AlScN

Stiff-cantilever dataset (k = 25 N/m, f0 = 270 kHz), 30 drive amplitudes
from 0.07 nm to 22.66 nm with 1000 points per FD curve.  Pre-baked workflow
from the v3 iteration of the Multi75 notebook:

* normalise by reference A0 so the simulator's natural scale matches
* pick 8 training + 5 held-out curves for speed
* anchor `d_offset` at the experimental `A_min` (not the 50% snap-in)
* multi-start per-curve `d_offset` to escape local minima
* loose damping prior so `gamma_lr` / `lambda_lr` can grow
* Q fixed (~400 for a Tap300 cantilever)
* evaluate on held-out without re-fitting shared physics


**⚠ v1 → v2 patch (ring-up bug fixed)**

The v1 run showed `Q` pinned at its lower bound (343 vs target 432), `C2`
scattered from 0.07 to 26 across adjacent training curves, `lambda_lr` blown
up to 226 nm, and held-out RMSE 10× worse than training (4.6 nm vs 0.46
nm).  Root cause: the simulator's `max_windows` was too low to let the
high-Q cantilever ring up to equilibrium amplitude.  At Q=432 the
cantilever needs ~635 cycles to reach 99% of A_eq from rest; the old
settings allowed only 40-160 cycles, leaving the simulator stuck at 68% of
equilibrium for every "first call" of every fit iteration.  The fitter
absorbed this systematic error into per-curve `C2, d_off`, and the held-out
evaluation couldn't compensate.

The fix is in cell 11 (longer `max_windows`).  Run cells in order — the
ring-up check just after will confirm `A_hat ≈ 1.0` before the actual
fits start.

## Imports

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['Arial']

import os
import time

# ------------------------------------------------------------------
# Physical constants for this cantilever
# ------------------------------------------------------------------
K_N_PER_M = 25.0
F0_HZ     = 270e3
Q_TARGET  = 200.0      # rough init - the global fit will refine
                       # (real intrinsic Q ~ 432, but the fit prefers ~90
                       # because additional dissipation gets absorbed here)

# ------------------------------------------------------------------
# Workflow knobs - change these to control how much fitting to do
# ------------------------------------------------------------------
N_TRAIN_CURVES     = 8       # number of curves used to fit shared physics
N_HELDOUT_CURVES   = 5       # disjoint, used for held-out evaluation
LOW_A_CUTOFF_FRAC  = 0.05    # drop the bottom 5% of A-points per curve
                              # (deep-contact regime where simple DMT fails
                              # and lock-in noise floor dominates)
DRIVE_MIN_NM       = 2.0     # ignore drives below this (poor SNR)

# Multi-start counts (more starts = more robust but slower)
N_STARTS_INDIVIDUAL = 3
N_STARTS_HELDOUT    = 5


## Load FD curves

In [ ]:
data = np.load('output/Tap300_AlScN.npz')
print('keys:', list(data.keys()))
for k in data.keys():
    print(f'  {k:<10} shape={data[k].shape}')

# Approach (1) and retract (2) - 1000 points each. Subsample 4x for speed.
height1 = data['height'][:, ::4]
amp1    = data['amp'][:, ::4]
phase1  = data['phase'][:, ::4]
height2 = data['height2'][:, ::4]
amp2    = data['amp2'][:, ::4]
phase2  = data['phase2'][:, ::4]
drive   = data['drive'][:-1]    # the drive array has one extra element

print(f'\n{len(drive)} drives from {drive.min():.2f} to {drive.max():.2f} nm')
print(f'{height1.shape[1]} points per curve after subsampling')


## Raw approach + retract curves

In [ ]:
cmap = plt.get_cmap('rainbow')
norm = mpl.colors.Normalize(vmin=drive.min(), vmax=drive.max())

fig, ax = plt.subplots(2, 2, figsize=(10, 6))
for i in range(len(drive)):
    c = cmap(norm(drive[i]))
    ax[0, 0].plot(height1[i], amp1[i],   '.-', lw=0.8, ms=1, color=c)
    ax[0, 1].plot(height1[i], phase1[i], '.-', lw=0.8, ms=1, color=c)
    ax[1, 0].plot(height2[i], amp2[i],   '.-', lw=0.8, ms=1, color=c)
    ax[1, 1].plot(height2[i], phase2[i], '.-', lw=0.8, ms=1, color=c)

ax[0, 0].set_title('Approach amplitude'); ax[0, 0].set_ylabel('A (nm)')
ax[0, 1].set_title('Approach phase');     ax[0, 1].set_ylabel('phi (deg)')
ax[1, 0].set_title('Retract amplitude');  ax[1, 0].set_xlabel('Height (nm)')
ax[1, 1].set_title('Retract phase');      ax[1, 1].set_xlabel('Height (nm)')
sm = mpl.cm.ScalarMappable(cmap=cmap, norm=norm); sm.set_array([])
fig.colorbar(sm, ax=ax.ravel().tolist(), label='Drive (nm)', pad=0.02)
plt.show()


## Set normalisation and build curve dicts

`conv_L = 1 / A0_ref` so that `A_hat = 1` corresponds to the free amplitude
at the reference (largest) drive.

In [ ]:
# Reference far-field amplitude from the largest-drive curve
j_ref = 0
n_far = max(3, int(0.10 * height1.shape[1]))
far_idx = np.argsort(height1[j_ref])[::-1][:n_far]
A0_ref_nm = float(np.median(amp1[j_ref, far_idx]))
conv_L = 1.0 / A0_ref_nm

print(f"Reference drive = {drive[j_ref]:.2f} nm")
print(f"A0_ref          = {A0_ref_nm:.3f} nm")
print(f"conv_L          = {conv_L:.5f} nm^-1   (so A_hat=1 means {A0_ref_nm:.2f} nm)")

# Drive scale factor check (D1 from the Multi75 notebook)
A0_all = np.array([
    float(np.median(amp1[j, np.argsort(height1[j])[::-1][:n_far]]))
    for j in range(len(drive))])
slope, intercept = np.polyfit(drive, A0_all, 1)
print(f"\nA0 = {slope:.4f} * drive + {intercept:+.3f}")
print(f"  drive scale alpha = {slope:.3f}   "
      f"(effective drive is {(1-slope)*100:.1f}% smaller than nominal)")


In [ ]:
from codes.spm_fd_rk45_extended_calibration import build_measured_fd_curves

curves_app = build_measured_fd_curves(
    drives_nm     = drive,
    height_nm     = height1,
    amplitude_nm  = amp1,
    phase_deg     = phase1,
    conv_L        = conv_L,
    A0_method     = "far_field_median",
    far_field_frac= 0.10,
    omega_drive_hat = 1.0,
    omega_ref_hat   = 1.0,
)

curves_ret = build_measured_fd_curves(
    drives_nm     = drive,
    height_nm     = height2,
    amplitude_nm  = amp2,
    phase_deg     = phase2,
    conv_L        = conv_L,
    A0_method     = "far_field_median",
    far_field_frac= 0.10,
    omega_drive_hat = 1.0,
    omega_ref_hat   = 1.0,
)
print(f"{len(curves_app)} approach + {len(curves_ret)} retract curves built")
print(f"Drives range: {curves_app[0]['drive_nm']:.2f} - {curves_app[-1]['drive_nm']:.2f} nm")


## RK45 options and free-air smoke test

In [ ]:
# ------------------------------------------------------------------
# IMPORTANT: ring-up cycles must exceed ~1.5*Q for high-Q cantilevers.
# For Q ~ 400, 99% of equilibrium amplitude takes ~635 cycles.
# Previously we used 40-160 cycles, which left the simulator stuck at
# 68% of equilibrium - the entire downstream fit chain was working
# against this systematic bias.  These options give 1000+ cycles for
# the first call (when the cantilever rings up from zero) and let
# warm-started subsequent calls converge quickly.
# ------------------------------------------------------------------

rk45_fit_options = dict(
    N_cycles=2, max_windows=500, settle_windows=10,    # was 20, 2
    tol_A=5e-4, tol_phi_deg=0.5, consecutive=2,
    demod_per_ref_cycle=24,
    rtol=1e-4, atol=1e-6,
    dt0=0.05, dt_min=1e-5, dt_max=0.25,
    max_steps_per_window=80000,
    phase_wrap_mode=360, use_warm_start=True,
)

rk45_plot_options = dict(
    N_cycles=4, max_windows=300, settle_windows=5,     # was 40, 3
    tol_A=1e-4, tol_phi_deg=0.05, consecutive=2,
    demod_per_ref_cycle=48,
    rtol=3e-6, atol=1e-8,
    dt0=0.02, dt_min=1e-5, dt_max=0.15,
    max_steps_per_window=200000,
    phase_wrap_mode=360, use_warm_start=True,
)

# Long-range damping defaults (lesson from Multi75 v3)
fixed_extra = dict(
    w_contact     = 0.05 * conv_L,
    gamma_lr      = 1e-3,
    lambda_lr     = 20.0 * conv_L,
    gamma_contact = 1e-3,
    omega_scale   = 1.0,
)


## Ring-up verification

Quick check that the simulator now reaches equilibrium amplitude at large
d (no surface interaction) within the allowed window count.  A_hat should
be very close to 1.0 across the whole far-field range.

In [ ]:
from codes.spm_fd_rk45_extended_calibration import simulate_curve_rk45_extended_python

C3_smk = curves_app[0]["A0_hat"] / Q_TARGET
# Wider far-field range to stress-test ring-up
d_test_hat = np.linspace(2.0, 10.0, 10)

print(f"Q_TARGET={Q_TARGET}  C3={C3_smk:.4e}  expected A_hat = C3*Q = "
      f"{C3_smk*Q_TARGET:.4f}")

A_sim_old, *_ = simulate_curve_rk45_extended_python(
    d_test_hat, C1=1e-5, C2=1.0, a0=0.05, Q=Q_TARGET, C3=C3_smk,
    omega_drive_hat=1.0, omega_ref_hat=1.0,
    w_contact=fixed_extra["w_contact"], gamma_lr=fixed_extra["gamma_lr"],
    lambda_lr=fixed_extra["lambda_lr"], gamma_contact=fixed_extra["gamma_contact"],
    # OLD settings, for comparison
    N_cycles=4, max_windows=40, settle_windows=3,
    tol_A=3e-5, tol_phi_deg=0.02, consecutive=2,
    demod_per_ref_cycle=48,
    rtol=3e-6, atol=1e-8,
    dt0=0.02, dt_min=1e-5, dt_max=0.15, max_steps_per_window=200000,
    phase_wrap_mode=360, use_warm_start=True,
)

A_sim_new, *_ = simulate_curve_rk45_extended_python(
    d_test_hat, C1=1e-5, C2=1.0, a0=0.05, Q=Q_TARGET, C3=C3_smk,
    omega_drive_hat=1.0, omega_ref_hat=1.0,
    w_contact=fixed_extra["w_contact"], gamma_lr=fixed_extra["gamma_lr"],
    lambda_lr=fixed_extra["lambda_lr"], gamma_contact=fixed_extra["gamma_contact"],
    **rk45_plot_options,   # NEW settings
)

print(f"\nOld opts  A_hat: {np.round(A_sim_old, 4)}")
print(f"New opts  A_hat: {np.round(A_sim_new, 4)}")
print(f"\nOld opts  mean far A_hat = {A_sim_old.mean():.4f}   (target 1.0)")
print(f"New opts  mean far A_hat = {A_sim_new.mean():.4f}   (target 1.0)")
if abs(A_sim_new.mean() - 1.0) > 0.02:
    print("\nWARNING: ring-up still incomplete.  Increase max_windows further.")
else:
    print("\nRing-up OK.  Proceed.")


## Phase-convention diagnostic

The lock-in's phase sign convention may differ from the simulator's
`-C3·cos(ωt)` reference.  Far-field auto-alignment only fixes a constant
shift, not a sign flip.  Try all 5 transformations on one well-fit curve
and pick the one with the smallest near-field residual.

In [ ]:
from codes.spm_fd_rk45_extended_calibration import (
    transform_phase_deg, estimate_phase_offset_far_field, angle_diff_period_deg
)

# Use the largest-drive curve (independent of subset selection downstream)
j_diag = 0      # biggest-drive curve in curves_app
c_diag = curves_app[j_diag]
A0_d   = float(c_diag["A0_hat"])

# Rough single-shot params - we just need a curve that's qualitatively right
A_sim, phi_sim, *_ = simulate_curve_rk45_extended_python(
    c_diag["d_hat"],
    C1=1e-4, C2=1.0, a0=0.1,
    Q=Q_TARGET, C3=c_diag["A0_hat"] / Q_TARGET,
    omega_drive_hat=1.0, omega_ref_hat=1.0,
    d_offset_hat=0.0,
    w_contact=fixed_extra["w_contact"], gamma_lr=fixed_extra["gamma_lr"],
    lambda_lr=fixed_extra["lambda_lr"], gamma_contact=fixed_extra["gamma_contact"],
    **rk45_plot_options,
)
print(f"Diagnostic curve: drive = {c_diag['drive_nm']:.2f} nm")
print(f"Sim raw phase range : [{phi_sim.min():.1f}, {phi_sim.max():.1f}]")
print(f"Exp phase range     : [{c_diag['phi_deg'].min():.1f}, "
      f"{c_diag['phi_deg'].max():.1f}]")

scores = {}
for mode in ("raw", "neg", "plus180", "minus_from_180", "minus_from_360"):
    phi_try = transform_phase_deg(phi_sim, mode=mode)
    offset = estimate_phase_offset_far_field(
        phi_try, c_diag["phi_deg"], c_diag["d_hat"],
        frac=0.15, period_deg=360.0)
    phi_aligned = phi_try + offset
    err = angle_diff_period_deg(phi_aligned, c_diag["phi_deg"], period_deg=360.0)
    rmse = float(np.sqrt(np.nanmean(err ** 2)))
    scores[mode] = (rmse, float(offset))
    print(f"  mode={mode:<18}  far-offset={offset:+7.2f}°  RMSE={rmse:6.2f}°")

best_mode = min(scores, key=lambda m: scores[m][0])
print(f"\nBest phase mode: {best_mode!r}  "
      f"(RMSE={scores[best_mode][0]:.2f}°, offset={scores[best_mode][1]:+.2f}°)")

# Plot the best vs raw to see the difference
import matplotlib.pyplot as plt
fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
ax[0].plot(c_diag["d_hat"]/conv_L, c_diag["phi_deg"], 'k.', ms=3, label='exp')
ax[0].plot(c_diag["d_hat"]/conv_L,
           phi_sim + scores["raw"][1], 'r-', lw=1, alpha=0.7,
           label=f"raw (RMSE={scores['raw'][0]:.1f}°)")
phi_b = transform_phase_deg(phi_sim, mode=best_mode) + scores[best_mode][1]
ax[0].plot(c_diag["d_hat"]/conv_L, phi_b, 'b-', lw=1.5,
           label=f"{best_mode} (RMSE={scores[best_mode][0]:.1f}°)")
ax[0].set_xlabel("Height (nm)"); ax[0].set_ylabel("Phase (deg)")
ax[0].set_title(f"Phase modes (drive={c_diag['drive_nm']:.1f} nm)")
ax[0].legend(fontsize=8)

ax[1].bar(range(len(scores)), [v[0] for v in scores.values()])
ax[1].set_xticks(range(len(scores))); ax[1].set_xticklabels(list(scores.keys()),
                                                              rotation=20)
ax[1].set_ylabel("Phase RMSE (deg)")
ax[1].set_title("Mode comparison")
plt.tight_layout(); plt.show()

# Bind for use by downstream cells
PHASE_MODE = best_mode


In [ ]:
from codes.spm_fd_rk45_extended_calibration import simulate_curve_rk45_extended_python

C3_init = curves_app[0]["A0_hat"] / Q_TARGET
print(f"Q={Q_TARGET}  C3={C3_init:.4e}  predicted A_hat = C3*Q = {C3_init*Q_TARGET:.3f}")

# Far-field simulation
d_test_hat = np.linspace(3.0, 6.0, 6)
A_sim, phi_sim, *_ = simulate_curve_rk45_extended_python(
    d_test_hat, C1=1e-5, C2=1.0, a0=0.05,
    Q=Q_TARGET, C3=C3_init,
    omega_drive_hat=1.0, omega_ref_hat=1.0,
    w_contact=fixed_extra["w_contact"],
    gamma_lr=fixed_extra["gamma_lr"],
    lambda_lr=fixed_extra["lambda_lr"],
    gamma_contact=fixed_extra["gamma_contact"],
    N_cycles=4, max_windows=40, settle_windows=3,
    tol_A=1e-5, tol_phi_deg=0.02, consecutive=2, demod_per_ref_cycle=48,
    phase_wrap_mode=360, use_warm_start=True,
)
print(f"Simulated A_hat (far): {np.round(A_sim, 4)}")
print(f"Mean far A_hat       : {A_sim.mean():.4f}   (target: ~ {curves_app[0]['A0_hat']:.3f})")


## Pick subset and held-out

Exclude very small drives (<2 nm) where the SNR is poor — those would
dominate the loss artificially.  Pick 8 training + 5 disjoint held-out
from the well-conditioned drives.

In [ ]:
# Apply DRIVE_MIN_NM filter and pick training/held-out per knobs
good_idx = [i for i in range(len(drive)) if drive[i] >= DRIVE_MIN_NM]
print(f"Using {len(good_idx)} drives >= {DRIVE_MIN_NM} nm: "
      f"{drive[good_idx[0]]:.2f} - {drive[good_idx[-1]]:.2f} nm")

if N_TRAIN_CURVES > len(good_idx):
    N_TRAIN_CURVES = len(good_idx)
sel_train = [good_idx[k] for k in
             np.linspace(0, len(good_idx)-1, N_TRAIN_CURVES).astype(int)]
remaining = [i for i in good_idx if i not in sel_train]
N_HELDOUT = min(N_HELDOUT_CURVES, len(remaining))
sel_eval = [remaining[k] for k in
            np.linspace(0, len(remaining)-1, N_HELDOUT).astype(int)] if N_HELDOUT else []
print(f"Train drives ({len(sel_train)}): {[round(drive[i], 1) for i in sel_train]}")
print(f"Eval  drives ({len(sel_eval)}): {[round(drive[i], 1) for i in sel_eval]}")

def subsample_curve(c, n=80):
    n0 = len(c["d_hat"])
    if n0 <= n: return c
    idx = np.linspace(0, n0-1, n).astype(int)
    return {**c, "d_hat": c["d_hat"][idx],
            "A_hat": c["A_hat"][idx], "phi_deg": c["phi_deg"][idx]}

def drop_lowest_A(c, frac=0.0):
    """Drop the `frac` fraction of points with smallest A_hat. Preserves
    original ordering of the remaining points."""
    if frac <= 0: return c
    A = c["A_hat"]; n_drop = int(np.ceil(frac * len(A)))
    if n_drop <= 0: return c
    order_A = np.argsort(A)
    keep_mask = np.ones(len(A), dtype=bool)
    keep_mask[order_A[:n_drop]] = False
    return {**c, "d_hat": c["d_hat"][keep_mask],
            "A_hat": c["A_hat"][keep_mask],
            "phi_deg": c["phi_deg"][keep_mask]}

curves_app_sub = [drop_lowest_A(subsample_curve(c, n=80), frac=LOW_A_CUTOFF_FRAC)
                  for c in curves_app]
curves_train = [curves_app_sub[i] for i in sel_train]
curves_eval  = [curves_app_sub[i] for i in sel_eval]

# Visualise what was dropped vs kept on one representative curve
fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
for jj, ax_i in enumerate(ax):
    j_show = sel_train[0] if jj == 0 else sel_train[-1]
    c_full = subsample_curve(curves_app[j_show], n=80)
    c_filt = drop_lowest_A(c_full, frac=LOW_A_CUTOFF_FRAC)
    kept_set = set(zip(c_filt["d_hat"], c_filt["A_hat"]))
    is_kept = np.array([(d, a) in kept_set
                        for d, a in zip(c_full["d_hat"], c_full["A_hat"])])
    ax_i.plot(c_full["d_hat"]/conv_L, c_full["A_hat"]/conv_L, 'r.', ms=4,
              label='dropped')
    ax_i.plot(c_full["d_hat"][is_kept]/conv_L,
              c_full["A_hat"][is_kept]/conv_L, 'b.', ms=4, label='kept')
    ax_i.axhline(LOW_A_CUTOFF_FRAC * curves_app[j_show]["A0_hat"] / conv_L,
                 color='gray', ls='--', lw=0.8,
                 label=f'A < {LOW_A_CUTOFF_FRAC*100:.0f}% of A0')
    ax_i.set_xlabel('Height (nm)'); ax_i.set_ylabel('A (nm)')
    ax_i.set_title(f"drive = {drive[j_show]:.2f} nm "
                   f"({len(c_filt['d_hat'])}/{len(c_full['d_hat'])} kept)")
    ax_i.legend(fontsize=8)
fig.tight_layout(); plt.show()

print(f"\nAfter filter: {len(curves_train[0]['d_hat'])} points / training curve "
      f"(was {len(subsample_curve(curves_app[sel_train[0]], n=80)['d_hat'])})")


## Surface position estimate (A_min anchor)

The cleanest anchor is the height at `A_min` — that's where the tip is
closest to the surface.  We use `-A_min` as the `d_offset` init: shifts
the experimental d so that A_min lands near d = 0 in simulator coords.

In [ ]:
def surface_estimates(d_hat, A_hat, A0):
    order = np.argsort(d_hat)[::-1]
    d_s = d_hat[order]; A_s = A_hat[order]
    j5 = np.where(A_s < 0.95 * A0)[0]
    h_5pct = float(d_s[j5[0]]) if len(j5) else float(d_s[-1])
    dAdd = np.gradient(A_s, d_s)
    h_grad = float(d_s[int(np.argmax(np.abs(dAdd)))])
    h_min = float(d_s[int(np.argmin(A_s))])
    return h_5pct, h_grad, h_min

est = np.array([surface_estimates(c["d_hat"], c["A_hat"], c["A0_hat"])
                for c in curves_app_sub])

fig, ax = plt.subplots(1, 1, figsize=(7, 3.5))
ax.plot(drive, est[:, 0]/conv_L, 'o-', label='95% A0 (interaction onset)')
ax.plot(drive, est[:, 1]/conv_L, 's-', label='max |dA/dd| (sharpest)')
ax.plot(drive, est[:, 2]/conv_L, '^-', label='A_min (deepest)')
ax.axhline(0, color='k', lw=0.5)
ax.set_xlabel("Drive (nm)"); ax.set_ylabel("Height (nm)")
ax.set_title("Three surface-position estimates")
ax.legend(); plt.tight_layout(); plt.show()

d_off_anchor = -est[:, 2]       # length len(drive), in hat units
print(f"d_off_anchor median = {np.median(d_off_anchor)/conv_L:+.2f} nm")
print(f"d_off_anchor std    = {d_off_anchor.std()/conv_L:.2f} nm")
print(f"d_off_anchor range  = [{d_off_anchor.min()/conv_L:+.2f}, "
      f"{d_off_anchor.max()/conv_L:+.2f}] nm")


## Multi-start individual fits on the training subset

3 starts per curve at `(-A_min, -A_min + 1 nm, -A_min - 2 nm)`.  Q fixed,
C1 / d_offset bounds wide.

In [ ]:
from codes.spm_fd_rk45_extended_calibration import (
    fit_one_fd_curve_rk45, make_single_x0
)

# Tighter Q bound than v3: avoid wild (C3, Q) bouncing per curve.
# Q can later move further in the GLOBAL fit (which sees all curves).
def bounds_v4(x0):
    lo = np.asarray(x0, float).copy(); hi = np.asarray(x0, float).copy()
    lo[0] -= 3.0*np.log(10.0); hi[0] += 3.0*np.log(10.0)   # log_C1 wide
    lo[1] -= 1.5*np.log(10.0); hi[1] += 1.5*np.log(10.0)   # log_C2 (was 2)
    lo[2] -= 1.5*np.log(10.0); hi[2] += 1.5*np.log(10.0)   # log_a0 (was 2)
    lo[3] -= 0.20*np.log(10.0); hi[3] += 0.20*np.log(10.0) # log_Q TIGHT
    lo[4] -= 3.0*np.log(10.0); hi[4] += 3.0*np.log(10.0)   # log_C3 wide
    lo[5] = -3.0*conv_L; hi[5] = +3.0*conv_L               # d_offset TIGHT (was 10)
    lo[6] = -180.0; hi[6] = +180.0
    return lo, hi

def fit_curve_multistart(curve, j_global, n_starts=N_STARTS_INDIVIDUAL):
    C1_init, C2_init, a0_init = 5e-4, 1.0, 0.1
    C3_init = curve["A0_hat"] / Q_TARGET
    anchor = d_off_anchor[j_global]
    if n_starts >= 5:
        starts = [anchor, anchor+0.5*conv_L, anchor-0.5*conv_L,
                  anchor+1.5*conv_L, anchor-1.5*conv_L]
    else:
        starts = [anchor, anchor + 1.0*conv_L, anchor - 1.0*conv_L][:n_starts]
    best = None
    for s in starts[:n_starts]:
        x0 = make_single_x0(C1=C1_init, C2=C2_init, a0=a0_init,
                            Q=Q_TARGET, C3=C3_init,
                            d_offset_hat=s, phi_offset_deg=0.0)
        bounds = bounds_v4(x0)
        try:
            fit = fit_one_fd_curve_rk45(
                curve, x0, bounds=bounds, fixed_extra=fixed_extra,
                w_amp=1.0, w_phi=0.03,             # gentle phase weight
                phase_mode=PHASE_MODE,
                far_field_phase_align=True, far_field_frac=0.15,
                phase_period_deg=360.0,
                use_amp_weighted_phase=True, sigma_phi0_deg=10.0,
                A_floor_frac=0.20,
                d_offset_prior_hat=2.0*conv_L, phi_offset_prior_deg=90.0,
                rk45_options=rk45_fit_options, max_nfev=80, verbose=0,
            )
            if best is None or fit["cost"] < best["cost"]:
                best = fit
        except Exception:
            pass
    return best

import time
t0 = time.time()
ind_fits = []
for k, j_global in enumerate(sel_train):
    fit = fit_curve_multistart(curves_app_sub[j_global], j_global,
                                n_starts=N_STARTS_INDIVIDUAL)
    ind_fits.append(fit)
    print(f"  [{k+1}/{len(sel_train)}] drive={fit['drive_nm']:6.2f}  "
          f"cost={fit['cost']:.3e}  Q={fit['Q']:6.1f}  "
          f"C2={fit['C2']:6.3f}  a0={fit['a0']:.4f}  "
          f"d_off={fit['d_offset_hat']/conv_L:+5.2f}nm  "
          f"phi_off={fit['phi_offset_deg']:+5.1f}°")
print(f"\nDone in {time.time()-t0:.1f} s")

# Outlier flag
median_cost = float(np.median([f['cost'] for f in ind_fits]))
print(f"\nMedian cost = {median_cost:.3e}")
for k, f in enumerate(ind_fits):
    if f['cost'] > 3.0 * median_cost:
        print(f"  outlier: drive={f['drive_nm']:.2f} nm  cost={f['cost']:.3e}  "
              f"({f['cost']/median_cost:.1f}x median)")


### Visualize 8 training fits

In [ ]:
from codes.spm_fd_rk45_extended_calibration import (
    transform_phase_deg, estimate_phase_offset_far_field,
)

fig, axes = plt.subplots(2, 4, figsize=(14, 6.5))
for ax, k in zip(axes.ravel(), range(len(ind_fits))):
    ax2 = ax.twinx()
    f = ind_fits[k]; c = curves_app[sel_train[k]]
    A_sim, phi_sim, *_ = simulate_curve_rk45_extended_python(
        c["d_hat"], C1=f["C1"], C2=f["C2"], a0=f["a0"], Q=f["Q"], C3=f["C3"],
        omega_drive_hat=1.0, omega_ref_hat=1.0,
        d_offset_hat=f["d_offset_hat"],
        w_contact=fixed_extra["w_contact"],
        gamma_lr=fixed_extra["gamma_lr"],
        lambda_lr=fixed_extra["lambda_lr"],
        gamma_contact=fixed_extra["gamma_contact"],
        **rk45_plot_options,
    )
    phi_use = transform_phase_deg(phi_sim, mode=PHASE_MODE)
    phi_auto = estimate_phase_offset_far_field(
        phi_use, c["phi_deg"], c["d_hat"], frac=0.15, period_deg=360.0)
    phi_use = phi_use + phi_auto + f.get("phi_offset_deg", 0.0)
    d_nm = c["d_hat"]/conv_L
    ax.plot(d_nm, c["A_hat"]/conv_L, 'b.', ms=2, alpha=0.5)
    ax.plot(d_nm, A_sim/conv_L, 'b-', lw=1.5)
    ax2.plot(d_nm, c["phi_deg"], 'r.', ms=2, alpha=0.5)
    ax2.plot(d_nm, phi_use, 'r-', lw=1.5)
    ax.set_title(f"d={f['drive_nm']:.2f}nm  cost={f['cost']:.2e}\n"
                 f"d_off={f['d_offset_hat']/conv_L:+.2f}nm", fontsize=8)
    ax.tick_params(labelsize=7); ax2.tick_params(labelsize=7)
fig.suptitle(f"Individual fits (phase_mode={PHASE_MODE!r})", y=1.0)
fig.tight_layout(); plt.show()


## Global fit on the 8-curve subset

Shared: C1, C2, a0, Q (fixed), damping.
Per-curve: C3, d_off, phi_off.  Loose damping prior (width 3.0).  Smoothness
regulariser on log_C3 across drives.

In [ ]:
from scipy.optimize import least_squares

n_sub = len(curves_train)

def unpack3(x):
    log_C1, log_C2, log_a0, log_Q = x[:4]
    log_wc, log_glr, log_llr, log_gcont, dom = x[4:9]
    log_C3 = x[9:9+n_sub]
    d_off  = x[9+n_sub:9+2*n_sub]
    p_off  = x[9+2*n_sub:9+3*n_sub]
    return dict(C1=np.exp(log_C1), C2=np.exp(log_C2), a0=np.exp(log_a0),
                Q=np.exp(log_Q),
                w_contact=np.exp(log_wc), gamma_lr=np.exp(log_glr),
                lambda_lr=np.exp(log_llr), gamma_contact=np.exp(log_gcont),
                domega=dom, C3=np.exp(log_C3), d_off=d_off, p_off=p_off)

m = lambda key: float(np.median([f[key] for f in ind_fits]))
Q_init_g = m("Q")
print(f"Initializing global fit with Q = {Q_init_g:.1f} (median of individuals)")

x0 = np.concatenate([
    np.array([np.log(m("C1")), np.log(m("C2")), np.log(m("a0")),
              np.log(Q_init_g),
              np.log(fixed_extra["w_contact"]),
              np.log(fixed_extra["gamma_lr"]),
              np.log(fixed_extra["lambda_lr"]),
              np.log(fixed_extra["gamma_contact"]),
              0.0]),
    np.log(np.array([f["C3"] for f in ind_fits])),
    np.array([f["d_offset_hat"] for f in ind_fits]),
    np.array([f["phi_offset_deg"] for f in ind_fits]),
])
lo = x0.copy(); hi = x0.copy()
lo[:3] -= 2.0*np.log(10.0); hi[:3] += 2.0*np.log(10.0)
lo[3]  -= 1.0*np.log(10.0); hi[3]  += 1.0*np.log(10.0)   # Q wide (global)
lo[4]  -= 2.0*np.log(10.0); hi[4]  += 2.0*np.log(10.0)
lo[5]  -= 1.5*np.log(10.0); hi[5]  += 1.5*np.log(10.0)
lo[6]  = np.log(3.0 * conv_L); hi[6] = np.log(40.0 * conv_L)   # lambda_lr 3-40 nm
lo[7]  -= 2.0*np.log(10.0); hi[7]  += 2.0*np.log(10.0)
lo[8] = -0.03; hi[8] = +0.03
lo[9:9+n_sub]    -= 1.5*np.log(10.0); hi[9:9+n_sub]    += 1.5*np.log(10.0)
lo[9+n_sub:9+2*n_sub] = -3.0*conv_L; hi[9+n_sub:9+2*n_sub] = +3.0*conv_L
lo[9+2*n_sub:] = -180.0; hi[9+2*n_sub:] = +180.0

def residual_g(x):
    p = unpack3(x); res = []
    omega_d = 1.0 + p["domega"]
    for k, c in enumerate(curves_train):
        A_sim, phi_sim, *_ = simulate_curve_rk45_extended_python(
            c["d_hat"], C1=p["C1"], C2=p["C2"], a0=p["a0"], Q=p["Q"], C3=p["C3"][k],
            omega_drive_hat=omega_d, omega_ref_hat=omega_d,
            d_offset_hat=p["d_off"][k],
            w_contact=p["w_contact"], gamma_lr=p["gamma_lr"],
            lambda_lr=p["lambda_lr"], gamma_contact=p["gamma_contact"],
            **rk45_fit_options,
        )
        A0 = float(c["A0_hat"])
        res.append((A_sim - c["A_hat"]) / A0)
        phi_use = transform_phase_deg(phi_sim, mode=PHASE_MODE)
        phi_auto = estimate_phase_offset_far_field(
            phi_use, c["phi_deg"], c["d_hat"], frac=0.15, period_deg=360.0)
        phi_use = phi_use + phi_auto + p["p_off"][k]
        dphi = (phi_use - c["phi_deg"] + 180.0) % 360.0 - 180.0
        sigma_phi = 10.0 * A0 / np.maximum(np.abs(c["A_hat"]), 0.20*A0)
        res.append(0.10 * dphi / sigma_phi)
    res.append(p["d_off"] / (1.0*conv_L))
    res.append(p["p_off"] / 30.0)                          # tighter (was 60)
    res.append(np.array([x[5]/3.0, x[6]/3.0, x[7]/3.0]))
    order = np.argsort([c["drive_nm"] for c in curves_train])
    diff2 = np.diff(np.diff(x[9:9+n_sub][order]))
    if len(diff2):
        res.append(0.5 * diff2)
    return np.concatenate(res)

print("\nStarting global subset fit ...")
t0 = time.time()
sol = least_squares(residual_g, x0, bounds=(lo, hi), loss="soft_l1",
                    f_scale=0.5, max_nfev=300, verbose=2)
print(f"Done in {time.time()-t0:.1f} s")
p_glob = unpack3(sol.x); p_glob["cost"] = sol.cost

print("\nShared parameters:")
for k in ("C1", "C2", "a0", "Q", "w_contact", "gamma_lr",
          "lambda_lr", "gamma_contact", "domega"):
    val = p_glob[k]
    extra = ""
    if k == "lambda_lr":
        extra = f"  (= {val/conv_L:.2f} nm)"
    if k == "w_contact":
        extra = f"  (= {val/conv_L:.3f} nm)"
    if k == "a0":
        extra = f"  (= {val/conv_L:.2f} nm)"
    print(f"  {k:<15} = {val}{extra}")

print(f"\nPer-curve d_off (nm): {[round(d/conv_L, 2) for d in p_glob['d_off']]}")
print(f"Per-curve phi_off(°): {[round(p, 1) for p in p_glob['p_off']]}")
print(f"Per-curve C3        : {[float(f'{c:.3e}') for c in p_glob['C3']]}")


### Global-fit overlays on the 8 training curves

In [ ]:
omega_base = 1.0 + p_glob["domega"]
fig, axes = plt.subplots(2, 4, figsize=(14, 6.5))
A_rmse_train = []
for ax, k in zip(axes.ravel(), range(n_sub)):
    ax2 = ax.twinx()
    c = curves_train[k]
    A_sim, phi_sim, *_ = simulate_curve_rk45_extended_python(
        c["d_hat"], C1=p_glob["C1"], C2=p_glob["C2"], a0=p_glob["a0"],
        Q=p_glob["Q"], C3=p_glob["C3"][k],
        omega_drive_hat=omega_base, omega_ref_hat=omega_base,
        d_offset_hat=p_glob["d_off"][k],
        w_contact=p_glob["w_contact"], gamma_lr=p_glob["gamma_lr"],
        lambda_lr=p_glob["lambda_lr"], gamma_contact=p_glob["gamma_contact"],
        **rk45_plot_options,
    )
    phi_use = transform_phase_deg(phi_sim, mode=PHASE_MODE)
    phi_auto = estimate_phase_offset_far_field(
        phi_use, c["phi_deg"], c["d_hat"], frac=0.15, period_deg=360.0)
    phi_use = phi_use + phi_auto + p_glob["p_off"][k]
    d_nm = c["d_hat"]/conv_L
    ax.plot(d_nm, c["A_hat"]/conv_L, 'b.', ms=2, alpha=0.5)
    ax.plot(d_nm, A_sim/conv_L, 'b-', lw=1.5)
    ax2.plot(d_nm, c["phi_deg"], 'r.', ms=2, alpha=0.5)
    ax2.plot(d_nm, phi_use, 'r-', lw=1.5)
    rmseA = float(np.sqrt(np.mean(((A_sim - c["A_hat"])/conv_L)**2)))
    A_rmse_train.append(rmseA)
    ax.set_title(f"drive={c['drive_nm']:.2f}  RMSE_A={rmseA:.2f}nm", fontsize=8)
    ax.tick_params(labelsize=7); ax2.tick_params(labelsize=7)
fig.suptitle(f"Tap300 subset global fit (mode={PHASE_MODE!r})  mean RMSE_A={np.mean(A_rmse_train):.2f}nm",
             y=1.0)
fig.tight_layout(); plt.show()
print(f"\nPer-curve training RMSE_A: {[round(x, 2) for x in A_rmse_train]}")
print(f"Mean: {np.mean(A_rmse_train):.3f} nm")


## Round-2 individual refit (lock shared physics, refit only C3 and d_off)

Round 1 individuals explored the (C3, Q) degeneracy basin-by-basin.  Now
that the global fit has chosen one Q value (and one set of shared physics),
re-fit each curve's `C3, d_off, phi_off` against the *shared* physics.  If
the round-2 individual costs are all close to the round-1 best curves
(~0.05–0.1), the shared physics is consistent with every curve.  If some
round-2 costs blow up, those curves disagree with the shared physics — a
red flag.

In [ ]:
# Lock shared physics from global; refit per-curve (C3, d_off, phi_off)
shared_v4 = dict(C1=p_glob["C1"], C2=p_glob["C2"], a0=p_glob["a0"],
                 Q=p_glob["Q"], w_contact=p_glob["w_contact"],
                 gamma_lr=p_glob["gamma_lr"],
                 lambda_lr=p_glob["lambda_lr"],
                 gamma_contact=p_glob["gamma_contact"])

def refit_perCurve(c, j_global):
    A0 = float(c["A0_hat"])
    def resid(x):
        log_C3, d_off, p_off = x
        A_sim, phi_sim, *_ = simulate_curve_rk45_extended_python(
            c["d_hat"], C1=shared_v4["C1"], C2=shared_v4["C2"], a0=shared_v4["a0"],
            Q=shared_v4["Q"], C3=np.exp(log_C3),
            omega_drive_hat=1.0, omega_ref_hat=1.0, d_offset_hat=d_off,
            w_contact=shared_v4["w_contact"], gamma_lr=shared_v4["gamma_lr"],
            lambda_lr=shared_v4["lambda_lr"],
            gamma_contact=shared_v4["gamma_contact"],
            **rk45_fit_options,
        )
        rA = (A_sim - c["A_hat"]) / A0
        phi_use = transform_phase_deg(phi_sim, mode=PHASE_MODE)
        phi_auto = estimate_phase_offset_far_field(
            phi_use, c["phi_deg"], c["d_hat"], frac=0.15, period_deg=360.0)
        phi_use = phi_use + phi_auto + p_off
        dphi = (phi_use - c["phi_deg"] + 180.0) % 360.0 - 180.0
        sigma = 10.0 * A0 / np.maximum(np.abs(c["A_hat"]), 0.20*A0)
        rP = 0.10 * dphi / sigma
        return np.concatenate([rA, rP])

    # Multi-start over d_off (which was the bimodal one)
    anchor = d_off_anchor[j_global]
    starts = [(np.log(c["A0_hat"]/shared_v4["Q"]), anchor, 0.0),
              (np.log(c["A0_hat"]/shared_v4["Q"]), anchor + 0.5*conv_L, 0.0),
              (np.log(c["A0_hat"]/shared_v4["Q"]), anchor - 0.5*conv_L, 0.0)]
    lo = np.array([np.log(c["A0_hat"]/shared_v4["Q"]) - 1.5*np.log(10),
                   -3.0*conv_L, -180.0])
    hi = np.array([np.log(c["A0_hat"]/shared_v4["Q"]) + 1.5*np.log(10),
                   +3.0*conv_L, +180.0])
    best = None
    for x0_ in starts:
        try:
            res = least_squares(resid, np.array(x0_), bounds=(lo, hi),
                                 loss="soft_l1", f_scale=0.5, max_nfev=40)
            if best is None or res.cost < best.cost:
                best = res
        except Exception:
            pass
    log_C3, d_off, p_off = best.x
    return dict(C3=float(np.exp(log_C3)), d_off=float(d_off),
                phi_off=float(p_off), cost=float(best.cost))

print("Round-2 individual refit (shared physics locked) ...")
t0 = time.time()
ind_fits_v4 = []
for k, j_global in enumerate(sel_train):
    r = refit_perCurve(curves_app_sub[j_global], j_global)
    ind_fits_v4.append(r)
    print(f"  [{k+1}/{len(sel_train)}] drive={drive[j_global]:6.2f}  "
          f"cost={r['cost']:.3e}  C3={r['C3']:.3e}  "
          f"d_off={r['d_off']/conv_L:+5.2f}nm  phi_off={r['phi_off']:+5.1f}°")
print(f"\nDone in {time.time()-t0:.1f} s")

# Compute per-curve RMSE
print(f"\n{'drive':>7} {'cost_r1':>9} {'cost_r2':>9} {'C3_r2':>10} "
      f"{'d_off (nm)':>10} {'phi_off':>8}")
for k, r in enumerate(ind_fits_v4):
    f1 = ind_fits[k]
    flag = "  ***" if r["cost"] > 3*np.median([rr["cost"] for rr in ind_fits_v4]) else ""
    print(f"{drive[sel_train[k]]:7.2f} {f1['cost']:9.3e} {r['cost']:9.3e} "
          f"{r['C3']:10.3e} {r['d_off']/conv_L:+10.2f} "
          f"{r['phi_off']:+8.1f}{flag}")


### Per-curve amplitude residual vs d (training set)

In [ ]:
# Plot (A_sim - A_exp) vs d for each training curve to see where the
# model deviates systematically.  Flat-near-zero residuals = good fit.
omega_base = 1.0 + p_glob["domega"]
fig, axes = plt.subplots(2, 4, figsize=(14, 6.5))
for ax, k in zip(axes.ravel(), range(len(curves_train))):
    c = curves_train[k]
    r = ind_fits_v4[k]
    A_sim, phi_sim, *_ = simulate_curve_rk45_extended_python(
        c["d_hat"],
        C1=p_glob["C1"], C2=p_glob["C2"], a0=p_glob["a0"],
        Q=p_glob["Q"], C3=r["C3"],
        omega_drive_hat=omega_base, omega_ref_hat=omega_base,
        d_offset_hat=r["d_off"],
        w_contact=p_glob["w_contact"], gamma_lr=p_glob["gamma_lr"],
        lambda_lr=p_glob["lambda_lr"], gamma_contact=p_glob["gamma_contact"],
        **rk45_plot_options,
    )
    d_nm = c["d_hat"]/conv_L
    resid_nm = (A_sim - c["A_hat"]) / conv_L
    ax.plot(d_nm, resid_nm, '.-', ms=3, lw=0.6)
    ax.axhline(0, color='k', lw=0.5)
    rmse = float(np.sqrt(np.mean(resid_nm**2)))
    ax.set_title(f"d={c['drive_nm']:.2f}  RMSE={rmse:.2f}nm", fontsize=8)
    ax.set_xlabel("Height (nm)", fontsize=7)
    ax.set_ylabel("A_sim - A_exp (nm)", fontsize=7)
    ax.tick_params(labelsize=7)
fig.suptitle("Per-curve amplitude residual (round-2 individuals)", y=1.0)
fig.tight_layout(); plt.show()


---

## Phase visualisation with constant offset removed

The constant per-curve phase offset (`phi_auto + p_off`) makes the raw
overlay hard to read.  Re-plot with both `phi_sim` and `phi_exp` shifted
to **zero in the far field**, so the y-axis now shows the
*interaction-induced* phase shift directly.  If the model is well-fit, the
two curves should overlay across the entire d range, not just at the
far-field anchor.

In [ ]:
omega_base = 1.0 + p_glob["domega"]

fig, axes = plt.subplots(2, 4, figsize=(14, 6.5))
for ax, k in zip(axes.ravel(), range(len(curves_train))):
    c = curves_train[k]
    r = ind_fits_v4[k]
    A_sim, phi_sim, *_ = simulate_curve_rk45_extended_python(
        c["d_hat"],
        C1=p_glob["C1"], C2=p_glob["C2"], a0=p_glob["a0"],
        Q=p_glob["Q"], C3=r["C3"],
        omega_drive_hat=omega_base, omega_ref_hat=omega_base,
        d_offset_hat=r["d_off"],
        w_contact=p_glob["w_contact"], gamma_lr=p_glob["gamma_lr"],
        lambda_lr=p_glob["lambda_lr"], gamma_contact=p_glob["gamma_contact"],
        **rk45_plot_options,
    )
    phi_use = transform_phase_deg(phi_sim, mode=PHASE_MODE)
    phi_auto = estimate_phase_offset_far_field(
        phi_use, c["phi_deg"], c["d_hat"], frac=0.15, period_deg=360.0)
    phi_use = phi_use + phi_auto + r["phi_off"]

    # Compute far-field reference for both (largest 15% of d)
    n_far = max(3, int(0.15 * len(c["d_hat"])))
    far_idx = np.argsort(c["d_hat"])[::-1][:n_far]
    phi_sim_far = float(np.mean(phi_use[far_idx]))
    phi_exp_far = float(np.mean(c["phi_deg"][far_idx]))

    # Subtract far-field so both start at zero
    phi_sim_rel = phi_use - phi_sim_far
    phi_exp_rel = c["phi_deg"] - phi_exp_far

    d_nm = c["d_hat"]/conv_L
    ax.plot(d_nm, phi_exp_rel, 'k.', ms=2.5, alpha=0.6, label='exp')
    ax.plot(d_nm, phi_sim_rel, 'r-', lw=1.5, label='sim')
    ax.axhline(0, color='gray', lw=0.5, ls='--')
    rmse_phi = float(np.sqrt(np.mean((phi_sim_rel - phi_exp_rel)**2)))
    ax.set_title(f"drive={c['drive_nm']:.2f}nm  "
                 f"RMSE_phi={rmse_phi:.1f}°", fontsize=8)
    ax.set_xlabel("Height (nm)", fontsize=8)
    ax.set_ylabel("phi - phi_far (deg)", fontsize=8)
    ax.tick_params(labelsize=7)
    if k == 0:
        ax.legend(fontsize=7)
fig.suptitle("Phase relative to far field (offset removed)", y=1.0)
fig.tight_layout(); plt.show()


## Held-out evaluation

Freeze shared physics, only refit `C3` and `d_off` per held-out curve.

In [ ]:
shared = dict(C1=p_glob["C1"], C2=p_glob["C2"], a0=p_glob["a0"], Q=p_glob["Q"],
              w_contact=p_glob["w_contact"], gamma_lr=p_glob["gamma_lr"],
              lambda_lr=p_glob["lambda_lr"], gamma_contact=p_glob["gamma_contact"])

def eval_one_heldout_multistart(c, j_global, n_starts=5):
    """Multi-start over d_off to escape local minima."""
    A0 = float(c["A0_hat"])
    def resid(x):
        log_C3, d_off = x
        A_sim, *_ = simulate_curve_rk45_extended_python(
            c["d_hat"], C1=shared["C1"], C2=shared["C2"], a0=shared["a0"],
            Q=shared["Q"], C3=np.exp(log_C3),
            omega_drive_hat=1.0, omega_ref_hat=1.0, d_offset_hat=d_off,
            w_contact=shared["w_contact"], gamma_lr=shared["gamma_lr"],
            lambda_lr=shared["lambda_lr"], gamma_contact=shared["gamma_contact"],
            **rk45_fit_options,
        )
        return (A_sim - c["A_hat"]) / A0
    anchor = d_off_anchor[j_global]
    inits = [anchor,
             anchor + 0.5*conv_L, anchor - 0.5*conv_L,
             anchor + 1.5*conv_L, anchor - 1.5*conv_L][:n_starts]
    best = None
    for d0 in inits:
        x0 = np.array([np.log(c["A0_hat"]/shared["Q"]), d0])
        lo = np.array([np.log(c["A0_hat"]/shared["Q"]) - 1.5*np.log(10), -5.0*conv_L])
        hi = np.array([np.log(c["A0_hat"]/shared["Q"]) + 1.5*np.log(10), +5.0*conv_L])
        try:
            res = least_squares(resid, x0, bounds=(lo, hi), loss="soft_l1",
                                f_scale=0.5, max_nfev=40, verbose=0)
            if best is None or res.cost < best.cost:
                best = res
        except Exception:
            pass
    log_C3, d_off = best.x
    A_sim, *_ = simulate_curve_rk45_extended_python(
        c["d_hat"], C1=shared["C1"], C2=shared["C2"], a0=shared["a0"],
        Q=shared["Q"], C3=np.exp(log_C3),
        omega_drive_hat=1.0, omega_ref_hat=1.0, d_offset_hat=d_off,
        w_contact=shared["w_contact"], gamma_lr=shared["gamma_lr"],
        lambda_lr=shared["lambda_lr"], gamma_contact=shared["gamma_contact"],
        **rk45_plot_options,
    )
    rmse_A = float(np.sqrt(np.mean(((A_sim - c["A_hat"])/conv_L)**2)))
    return dict(C3=float(np.exp(log_C3)), d_off=float(d_off),
                A_sim=A_sim, rmse_A=rmse_A)

print("Evaluating held-out (multi-start over d_off) ...")
heldout = []
fig, axes = plt.subplots(1, 5, figsize=(16, 3.4))
for ax, j_global in zip(axes.ravel(), sel_eval):
    c = curves_app_sub[j_global]
    r = eval_one_heldout_multistart(c, j_global, n_starts=5)
    heldout.append((j_global, r))
    d_nm = c["d_hat"]/conv_L
    ax.plot(d_nm, c["A_hat"]/conv_L, 'b.', ms=2, label='exp')
    ax.plot(d_nm, r["A_sim"]/conv_L, 'b-', lw=1.5, label='predict')
    ax.set_title(f"drive={c['drive_nm']:.2f}\nRMSE={r['rmse_A']:.2f}nm",
                 fontsize=8)
    ax.set_xlabel("h (nm)", fontsize=8); ax.tick_params(labelsize=7)
    if ax is axes[0]:
        ax.set_ylabel("A (nm)", fontsize=8); ax.legend(fontsize=7)
fig.suptitle("Tap300 held-out predictions (multi-start)", y=1.05)
fig.tight_layout(); plt.show()

print(f"\nHeld-out drive(nm)  d_off(nm)   C3       RMSE_A(nm)")
for j_global, r in heldout:
    print(f"          {drive[j_global]:6.2f}    {r['d_off']/conv_L:+6.2f}   "
          f"{r['C3']:.3e}  {r['rmse_A']:.3f}")
print(f"\nMean held-out RMSE: "
      f"{np.mean([r['rmse_A'] for _, r in heldout]):.3f} nm")


## Physical-units summary

In [ ]:
omega0   = 2*np.pi*F0_HZ
A_ref_m  = A0_ref_nm * 1e-9
F_unit_N = K_N_PER_M * A_ref_m

print(f"Cantilever: k = {K_N_PER_M} N/m, f0 = {F0_HZ/1e3:.0f} kHz, "
      f"Q_fit = {p_glob['Q']:.1f}")
print(f"Length unit: 1 hat = {1/conv_L:.3f} nm   "
      f"(= reference A0)")
print(f"Force unit : 1 hat = {F_unit_N*1e9:.3f} nN   (= k * A_ref)")
print()
print("Long-range dissipation:")
print(f"  gamma_lr      = {p_glob['gamma_lr']:.3e}   "
      f"(damping fraction near contact)")
print(f"  lambda_lr     = {p_glob['lambda_lr']/conv_L:.2f} nm   "
      f"(decay length)")
print(f"  gamma_contact = {p_glob['gamma_contact']:.3e}")
print()
print("Conservative force-law parameters:")
F_attr_1hat = p_glob['C1'] / (1.05)**2 * F_unit_N * 1e9
print(f"  C1 = {p_glob['C1']:.3e}  (F_attr at gap=1 hat  =  {F_attr_1hat:.3f} nN)")
print(f"  C2 = {p_glob['C2']:.3f}  (Hertz stiffness)")
print(f"  a0 = {p_glob['a0']/conv_L:.3f} nm  (short-range cutoff)")
print(f"  w_contact = {p_glob['w_contact']/conv_L:.3f} nm")
print()
print(f"Effective Q:")
print(f"  far field  : {1/(1/p_glob['Q'] + p_glob['gamma_lr']):.1f}")
print(f"  in contact : {1/(1/p_glob['Q'] + p_glob['gamma_contact']):.1f}")
print()
print(f"Drive scale factor alpha (D1) = {slope:.3f}")


## Scale-up to all curves

If the subset fits are good (held-out RMSE < 1 nm), redo the global fit
with all 30 curves by changing `sel_train` to include all of `good_idx`
and re-running the global-fit cell.  Expect ~5-8 min for the full 30-curve
fit on this dataset.

If they're not good, the diagnostics to look at first are:
* `lambda_lr` — if it sits at its upper bound (~150 nm), the interaction
  reach is even longer; use the LJ or Morse force model from
  `spm_force_models_branches`.
* per-curve `d_off` from cell V3.3 output — if some are bimodal, the
  multi-start needs more starts (raise `n_starts=5`).
* phase residual — if the simulated phase has the wrong sign of deviation,
  try `phase_mode="minus_from_360"` in the residual function.

## 4.5  Targeted branch-aware refit (phase-flip recovery)

For curves where the individual fit ends up with the *opposite* phase
trend from the experiment (e.g., drive ≈ 11.37 nm), the optimizer is
locked into the wrong dynamical branch — net-attractive vs. net-repulsive
— and local minimisation cannot escape it.  Two tools help, in this order:

1. **Sparse grid over `(C2, a0, d_offset)`** — these parameters select
   which branch the simulator settles on.  A 3 × 3 × 4 grid (36 starts)
   from log-spaced offsets is enough to find the right basin.

2. **Contact-region weights** — once the basin is correct, multiply each
   amplitude residual by `(1 + α·|A/A0 − 1|)` and each phase residual by
   `(1 + β·|φ − φ_free|/90°)`.  This pushes the loss budget into the
   contact region instead of the trivially-matching far field.

Edit `PROBLEM_DRIVES_NM` below if other curves also need the recovery.


In [ ]:
# ---------------------------------------------------------------
# Pick drives to recover, then build branch-aware multi-start fits.
# ---------------------------------------------------------------
from codes.spm_fd_rk45_extended_calibration import (
    fit_one_fd_curve_rk45, make_single_x0,
    simulate_curve_rk45_extended_python,
    transform_phase_deg, estimate_phase_offset_far_field,
)

PROBLEM_DRIVES_NM = [11.37]    # add more drives here if needed
PROBLEM_TOL_NM    = 0.5         # match tolerance


def _which_indices(target_drives, tol=PROBLEM_TOL_NM):
    out = []
    for td in target_drives:
        diffs = np.abs(drive - td)
        if diffs.min() <= tol:
            out.append(int(np.argmin(diffs)))
    return sorted(set(out))


problem_idx = _which_indices(PROBLEM_DRIVES_NM)
print(f"Recovering drives: {[round(drive[i], 2) for i in problem_idx]}")

# Show that the original individual fit indeed has a wrong-branch
# phase: compare median phase shift far-from-vs-near-surface.
def _branch_diag(ind_fit, c):
    A_sim, phi_sim, *_ = simulate_curve_rk45_extended_python(
        c["d_hat"], C1=ind_fit["C1"], C2=ind_fit["C2"], a0=ind_fit["a0"],
        Q=ind_fit["Q"], C3=ind_fit["C3"],
        omega_drive_hat=1.0, omega_ref_hat=1.0,
        d_offset_hat=ind_fit["d_offset_hat"],
        w_contact=fixed_extra["w_contact"],
        gamma_lr=fixed_extra["gamma_lr"],
        lambda_lr=fixed_extra["lambda_lr"],
        gamma_contact=fixed_extra["gamma_contact"],
        **rk45_plot_options,
    )
    phi_use = transform_phase_deg(phi_sim, mode=PHASE_MODE)
    phi_auto = estimate_phase_offset_far_field(
        phi_use, c["phi_deg"], c["d_hat"], frac=0.15, period_deg=360.0,
    )
    phi_use = phi_use + phi_auto + ind_fit.get("phi_offset_deg", 0.0)

    far_mask = c["d_hat"] >= np.percentile(c["d_hat"], 85)
    near_mask = c["d_hat"] <= np.percentile(c["d_hat"], 20)
    delta_exp = float(np.nanmedian(c["phi_deg"][near_mask])
                       - np.nanmedian(c["phi_deg"][far_mask]))
    delta_sim = float(np.nanmedian(phi_use [near_mask])
                       - np.nanmedian(phi_use [far_mask]))
    return delta_exp, delta_sim


print()
print("  drive   delta_phi_exp   delta_phi_sim   flipped?")
print("  -----   -------------   -------------   --------")
for j in problem_idx:
    k = sel_train.index(j) if j in sel_train else None
    if k is None:
        print(f"  {drive[j]:5.2f}   (not in training set)")
        continue
    de, ds = _branch_diag(ind_fits[k], curves_app[j])
    flipped = "YES" if de * ds < 0 else "no "
    print(f"  {drive[j]:5.2f}     {de:+7.2f}        {ds:+7.2f}       {flipped}")


In [ ]:
# ---------------------------------------------------------------
# Branch-aware fit: sparse grid + contact-region residual weights.
# ---------------------------------------------------------------
import time
from scipy.optimize import least_squares

# Sparse grid in (C2, a0, d_offset) — these select the dynamical branch.
GRID_C2     = np.array([0.1, 1.0, 10.0])               # contact stiffness
GRID_a0     = np.array([0.03, 0.10, 0.30])             # attraction depth
GRID_d_off  = np.array([-2.0, -0.8, +0.8, +2.0]) * conv_L   # surface offset (hat)

# Contact-region weights: emphasise points where (A, phi) deviate from
# their free-air values, which is where the physics actually lives.
ALPHA_AMP  = 4.0    # multiplier for |A/A0 - 1|
BETA_PHI   = 4.0    # multiplier for |phi - phi_far| / 90 deg


def _aligned_phase_sim(c, p_fit):
    """Simulate at p_fit, return phi after far-field alignment + offset."""
    A_sim, phi_sim, *_ = simulate_curve_rk45_extended_python(
        c["d_hat"], C1=p_fit["C1"], C2=p_fit["C2"], a0=p_fit["a0"],
        Q=p_fit["Q"], C3=p_fit["C3"],
        omega_drive_hat=1.0, omega_ref_hat=1.0,
        d_offset_hat=p_fit["d_offset_hat"],
        w_contact=fixed_extra["w_contact"],
        gamma_lr=fixed_extra["gamma_lr"],
        lambda_lr=fixed_extra["lambda_lr"],
        gamma_contact=fixed_extra["gamma_contact"],
        **rk45_fit_options,
    )
    phi_use = transform_phase_deg(phi_sim, mode=PHASE_MODE)
    phi_off = estimate_phase_offset_far_field(
        phi_use, c["phi_deg"], c["d_hat"], frac=0.15, period_deg=360.0,
    )
    return A_sim, phi_use + phi_off + p_fit.get("phi_offset_deg", 0.0)


def _contact_weighted_cost(p_fit, c):
    """Weighted SSE: amp residuals weighted by |A/A0 - 1|, phi residuals
    by |phi_exp - phi_far| / 90 deg.  Plus a small uniform floor."""
    try:
        A_sim, phi_sim = _aligned_phase_sim(c, p_fit)
    except Exception:
        return np.inf

    A_exp   = c["A_hat"];   phi_exp = c["phi_deg"];   A0 = c["A0_hat"]
    far_mask = c["d_hat"] >= np.percentile(c["d_hat"], 85)
    phi_far  = float(np.nanmedian(phi_exp[far_mask]))

    w_A   = 1.0 + ALPHA_AMP * np.abs(A_exp / max(A0, 1e-9) - 1.0)
    w_phi = 1.0 + BETA_PHI  * np.abs(phi_exp - phi_far) / 90.0

    r_A   = (A_sim - A_exp) / max(A0, 1e-9)
    r_phi = ((phi_sim - phi_exp + 180.0) % 360.0 - 180.0) / 90.0
    cost  = float(np.nansum((w_A * r_A) ** 2)
                  + 0.03 * np.nansum((w_phi * r_phi) ** 2))
    return cost


def sparse_grid_refit(j_curve, n_local_starts=2):
    """For one drive index, sweep the sparse (C2, a0, d_off) grid using
    fit_one_fd_curve_rk45 with default loss to localise candidates,
    then pick the best by contact-weighted cost."""
    c = curves_app_sub[j_curve]
    A0 = c["A0_hat"]
    C1_init = 5e-4
    C3_init = A0 / Q_TARGET

    best  = None
    n_tot = len(GRID_C2) * len(GRID_a0) * len(GRID_d_off)
    print(f"\nSparse grid for drive = {drive[j_curve]:.2f} nm  ({n_tot} starts)")
    t0 = time.time()

    for C2_g in GRID_C2:
        for a0_g in GRID_a0:
            for d_g in GRID_d_off:
                x0 = make_single_x0(
                    C1=C1_init, C2=C2_g, a0=a0_g, Q=Q_TARGET, C3=C3_init,
                    d_offset_hat=d_g, phi_offset_deg=0.0,
                )
                lo = np.asarray(x0, float).copy(); hi = lo.copy()
                lo[0] -= 3.0*np.log(10.0); hi[0] += 3.0*np.log(10.0)
                lo[1] -= 1.5*np.log(10.0); hi[1] += 1.5*np.log(10.0)
                lo[2] -= 1.5*np.log(10.0); hi[2] += 1.5*np.log(10.0)
                lo[3] -= 0.20*np.log(10.0); hi[3] += 0.20*np.log(10.0)
                lo[4] -= 3.0*np.log(10.0); hi[4] += 3.0*np.log(10.0)
                lo[5] = -3.0*conv_L; hi[5] = +3.0*conv_L
                lo[6] = -180.0; hi[6] = +180.0

                try:
                    fit = fit_one_fd_curve_rk45(
                        c, x0, bounds=(lo, hi), fixed_extra=fixed_extra,
                        w_amp=1.0, w_phi=0.10,            # higher phase weight
                        phase_mode=PHASE_MODE,
                        far_field_phase_align=True, far_field_frac=0.15,
                        phase_period_deg=360.0,
                        use_amp_weighted_phase=True, sigma_phi0_deg=10.0,
                        A_floor_frac=0.20,
                        d_offset_prior_hat=2.0*conv_L,
                        phi_offset_prior_deg=90.0,
                        rk45_options=rk45_fit_options, max_nfev=40, verbose=0,
                    )
                except Exception:
                    continue

                fit["drive_nm"] = float(drive[j_curve])
                # Re-score with contact-weighted cost
                cw_cost = _contact_weighted_cost(fit, c)
                fit["cw_cost"] = cw_cost
                if (best is None) or (cw_cost < best["cw_cost"]):
                    best = fit

    elapsed = time.time() - t0
    print(f"  done in {elapsed:.1f} s,  best cw_cost = {best['cw_cost']:.3e}")
    return best


# Run the refit on every problem drive and stash in `refits`.
refits = {}
for j in problem_idx:
    refits[j] = sparse_grid_refit(j)


In [ ]:
# ---------------------------------------------------------------
# Compare original individual fit vs. branch-aware refit.
# ---------------------------------------------------------------
fig, axes = plt.subplots(len(problem_idx), 2, figsize=(11, 3.5 * len(problem_idx)),
                          squeeze=False)
for row, j in enumerate(problem_idx):
    c   = curves_app[j]
    d_nm = c["d_hat"] / conv_L

    # Original individual fit (if in training set), else fall back to
    # a fresh run with default initial conditions.
    if j in sel_train:
        f_orig = ind_fits[sel_train.index(j)]
    else:
        f_orig = None

    f_new  = refits[j]

    # Amplitude panel
    axA = axes[row, 0]
    axA.plot(d_nm, c["A_hat"] / conv_L, "ko", ms=2.5, mfc="white",
              mew=0.6, label="experiment")
    if f_orig is not None:
        A_o, _ = _aligned_phase_sim(c, f_orig)
        axA.plot(d_nm, A_o / conv_L, "-", color="tab:orange",
                  lw=1.1, label="orig. ind. fit")
    A_n, _ = _aligned_phase_sim(c, f_new)
    axA.plot(d_nm, A_n / conv_L, "-", color="tab:blue",
              lw=1.4, label="branch-aware refit")
    axA.set_ylabel("Amplitude (nm)")
    axA.set_title(f"drive = {drive[j]:.2f} nm — amplitude", fontsize=9)
    axA.legend(frameon=False, fontsize=8)

    # Phase panel
    axP = axes[row, 1]
    axP.plot(d_nm, c["phi_deg"], "ko", ms=2.5, mfc="white",
              mew=0.6, label="experiment")
    if f_orig is not None:
        _, phi_o = _aligned_phase_sim(c, f_orig)
        axP.plot(d_nm, phi_o, "-", color="tab:orange",
                  lw=1.1, label="orig. ind. fit")
    _, phi_n = _aligned_phase_sim(c, f_new)
    axP.plot(d_nm, phi_n, "-", color="tab:blue",
              lw=1.4, label="branch-aware refit")
    axP.set_ylabel("Phase (deg)")
    axP.set_title(f"drive = {drive[j]:.2f} nm — phase", fontsize=9)
    axP.legend(frameon=False, fontsize=8)

    for ax in (axA, axP):
        ax.set_xlabel("Tip-sample distance (nm)")

fig.tight_layout()
plt.show()
print("\nbranch-aware refit results stored in `refits`")
print("If the new phase trend matches the experiment, push these as the")
print("initial conditions for the next round of the global fit.")


## 4.6  Propagate the refit into the global fit (and Fig. 2)

The branch-aware refit from Section 4.5 only changes the *individual*
fit for the problem drive.  Fig. 2's phase panel still shows the wrong
branch because the global fit shares `(C2, a0)` across all training
curves and the old `ind_fits` median seeds the global solver in the
wrong basin.

Two attempts are made here, in order:

**Step A — Re-seeded global fit (shared physics preserved).**
Replace the entry in `ind_fits` for the problem drive with the
branch-aware refit, recompute the median seed, re-run the same global
fit, and regenerate the (a)/(b) panels.  Best case: a single
`(C2, a0)` pair satisfies all training curves and the 11.37 nm phase
stops flipping.

**Step B — Per-curve C2/a0 (relaxed shared physics).**
If step A still flips, the dataset truly spans the attractive /
repulsive bistability and one shared `(C2, a0)` cannot satisfy both
ends.  In that case we allow `C2` and `a0` to vary per curve (the
remaining shared parameters — Q, w_contact, gamma_lr, lambda_lr,
gamma_contact, domega — stay shared) and re-run.


In [ ]:
# ---------------------------------------------------------------
# Step A: re-seeded global fit with shared physics preserved.
# Defensive against missing refit entries and collapsed medians.
# ---------------------------------------------------------------
import copy
from scipy.optimize import least_squares


def _safe_log(v, floor=1e-12):
    """log of max(v, floor); avoids a collapsed median crashing the seed."""
    return float(np.log(max(float(v), floor)))


def _safe_get(d, key, fallback):
    v = d.get(key, None) if isinstance(d, dict) else None
    if v is None or not np.isfinite(v):
        return fallback
    return float(v)


def _resolve_per_curve(val, k):
    """Return val[k] if val is array-like, else float(val)."""
    if np.ndim(val) == 0:
        return float(val)
    return float(np.asarray(val).ravel()[k])


# 1.  Patch ind_fits with the branch-aware refit where available.
ind_fits_patched = copy.deepcopy(ind_fits)
patched_indices  = []
skipped          = []
for j in problem_idx:
    if j not in sel_train:
        skipped.append((j, "not in training set"))
        continue
    k = sel_train.index(j)
    r = refits.get(j, None)
    if r is None:
        skipped.append((j, "refits[j] is None — sparse grid failed"))
        continue
    # Make sure the refit carries every key the global solver expects.
    needed = ("C1", "C2", "a0", "Q", "C3", "d_offset_hat", "phi_offset_deg")
    missing = [n for n in needed if n not in r]
    if missing:
        # Fill missing slots from the original ind_fits entry.
        for n in missing:
            r[n] = ind_fits_patched[k].get(n, 0.0 if n.endswith("offset_deg") else 1.0)
    ind_fits_patched[k] = r
    patched_indices.append(k)

print(f"Patched ind_fits entries: {patched_indices} "
      f"(drives {[round(drive[sel_train[k]], 2) for k in patched_indices]})")
for j, why in skipped:
    print(f"  skipped drive {drive[j]:.2f}: {why}")


# 2.  Build the global seed (same parameterisation as cell 27).
n_sub = len(curves_train)


def _unpack3(x):
    log_C1, log_C2, log_a0, log_Q = x[:4]
    log_wc, log_glr, log_llr, log_gcont, dom = x[4:9]
    log_C3 = x[9:9+n_sub]
    d_off  = x[9+n_sub:9+2*n_sub]
    p_off  = x[9+2*n_sub:9+3*n_sub]
    return dict(C1=np.exp(log_C1), C2=np.exp(log_C2), a0=np.exp(log_a0),
                Q=np.exp(log_Q),
                w_contact=np.exp(log_wc), gamma_lr=np.exp(log_glr),
                lambda_lr=np.exp(log_llr), gamma_contact=np.exp(log_gcont),
                domega=dom, C3=np.exp(log_C3), d_off=d_off, p_off=p_off)


def _safe_median(key, fits, default=1.0):
    vals = [float(f.get(key, np.nan)) for f in fits]
    vals = [v for v in vals if np.isfinite(v) and v > 0]
    return float(np.median(vals)) if vals else float(default)


C1_seed = _safe_median("C1", ind_fits_patched, default=5e-4)
C2_seed = _safe_median("C2", ind_fits_patched, default=1.0)
a0_seed = _safe_median("a0", ind_fits_patched, default=0.1)
Q_seed  = _safe_median("Q",  ind_fits_patched, default=Q_TARGET)
print(f"seed (C1, C2, a0, Q) = "
      f"({C1_seed:.3e}, {C2_seed:.3e}, {a0_seed:.3e}, {Q_seed:.2f})")

x0_A = np.concatenate([
    np.array([_safe_log(C1_seed), _safe_log(C2_seed), _safe_log(a0_seed),
              _safe_log(Q_seed),
              _safe_log(fixed_extra["w_contact"]),
              _safe_log(fixed_extra["gamma_lr"]),
              _safe_log(fixed_extra["lambda_lr"]),
              _safe_log(fixed_extra["gamma_contact"]),
              0.0]),
    np.array([_safe_log(_safe_get(f, "C3", 1.0)) for f in ind_fits_patched]),
    np.array([_safe_get(f, "d_offset_hat",  0.0) for f in ind_fits_patched]),
    np.array([_safe_get(f, "phi_offset_deg", 0.0) for f in ind_fits_patched]),
])

lo_A = x0_A.copy(); hi_A = x0_A.copy()
lo_A[:3] -= 2.0*np.log(10.0); hi_A[:3] += 2.0*np.log(10.0)
lo_A[3]  -= 1.0*np.log(10.0); hi_A[3]  += 1.0*np.log(10.0)
lo_A[4]  -= 2.0*np.log(10.0); hi_A[4]  += 2.0*np.log(10.0)
lo_A[5]  -= 1.5*np.log(10.0); hi_A[5]  += 1.5*np.log(10.0)
lo_A[6]  = np.log(3.0 * conv_L); hi_A[6] = np.log(40.0 * conv_L)
lo_A[7]  -= 2.0*np.log(10.0); hi_A[7]  += 2.0*np.log(10.0)
lo_A[8]  = -0.03; hi_A[8] = +0.03
lo_A[9:9+n_sub]    -= 1.5*np.log(10.0); hi_A[9:9+n_sub]    += 1.5*np.log(10.0)
lo_A[9+n_sub:9+2*n_sub] = -3.0*conv_L; hi_A[9+n_sub:9+2*n_sub] = +3.0*conv_L
lo_A[9+2*n_sub:] = -180.0; hi_A[9+2*n_sub:] = +180.0


# 3.  Unified residual that handles BOTH scalar (Step A) and per-curve
#     (Step B) C2 / a0 via _resolve_per_curve.
def _residual_global(x, _unpack):
    try:
        p = _unpack(x)
    except Exception:
        return np.full(2 * n_sub * 60, 1e3)
    omega_d = 1.0 + p["domega"]
    res = []
    for k, c in enumerate(curves_train):
        try:
            A_sim, phi_sim, *_ = simulate_curve_rk45_extended_python(
                c["d_hat"],
                C1=float(p["C1"]),
                C2=_resolve_per_curve(p["C2"], k),
                a0=_resolve_per_curve(p["a0"], k),
                Q=float(p["Q"]),
                C3=_resolve_per_curve(p["C3"], k),
                omega_drive_hat=omega_d, omega_ref_hat=1.0,
                d_offset_hat=_resolve_per_curve(p["d_off"], k),
                w_contact=float(p["w_contact"]),
                gamma_lr=float(p["gamma_lr"]),
                lambda_lr=float(p["lambda_lr"]),
                gamma_contact=float(p["gamma_contact"]),
                **rk45_fit_options,
            )
            phi_use = transform_phase_deg(phi_sim, mode=PHASE_MODE)
            phi_off = estimate_phase_offset_far_field(
                phi_use, c["phi_deg"], c["d_hat"],
                frac=0.15, period_deg=360.0,
            )
            phi_use = phi_use + phi_off + _resolve_per_curve(p["p_off"], k)
            r_A   = (A_sim - c["A_hat"]) / max(c["A0_hat"], 1e-9)
            r_phi = ((phi_use - c["phi_deg"] + 180.0) % 360.0 - 180.0) / 90.0
            res.append(np.nan_to_num(r_A,   nan=10.0, posinf=10.0, neginf=-10.0))
            res.append(0.03 * np.nan_to_num(r_phi, nan=10.0, posinf=10.0, neginf=-10.0))
        except Exception:
            # Penalty residual on this curve so the optimizer learns to
            # stay away from regions that crash the simulator.
            res.append(np.full_like(c["d_hat"], 10.0))
            res.append(np.full_like(c["d_hat"], 10.0))
    return np.concatenate(res)


print("\nRunning Step A global re-fit (this can take a few minutes) ...")
t0 = time.time()
res_A = least_squares(
    _residual_global, x0_A, bounds=(lo_A, hi_A),
    args=(_unpack3,),
    method="trf", max_nfev=400, x_scale="jac", loss="soft_l1",
    verbose=2,
)
p_glob_A = _unpack3(res_A.x)
print(f"Step A done in {time.time()-t0:.1f} s,  cost = {res_A.cost:.3e}")
print(f"  shared C2 = {p_glob_A['C2']:.3e}   shared a0 = {p_glob_A['a0']:.3e}")


In [ ]:
# ---------------------------------------------------------------
# Step A — Compare experiment, original Fig. 2 global fit, and the
# re-seeded global fit at each problem drive.
# ---------------------------------------------------------------
def _aligned_phase_sim_safe(c, p_fit):
    """Same as _aligned_phase_sim but tolerates per-curve C2/a0."""
    A_sim, phi_sim, *_ = simulate_curve_rk45_extended_python(
        c["d_hat"],
        C1=float(p_fit["C1"]),
        C2=float(p_fit["C2"]),
        a0=float(p_fit["a0"]),
        Q=float(p_fit["Q"]),
        C3=float(p_fit["C3"]),
        omega_drive_hat=1.0, omega_ref_hat=1.0,
        d_offset_hat=float(p_fit["d_offset_hat"]),
        w_contact=fixed_extra["w_contact"],
        gamma_lr=fixed_extra["gamma_lr"],
        lambda_lr=fixed_extra["lambda_lr"],
        gamma_contact=fixed_extra["gamma_contact"],
        **rk45_plot_options,
    )
    phi_use = transform_phase_deg(phi_sim, mode=PHASE_MODE)
    phi_off = estimate_phase_offset_far_field(
        phi_use, c["phi_deg"], c["d_hat"], frac=0.15, period_deg=360.0,
    )
    return A_sim, phi_use + phi_off + float(p_fit.get("phi_offset_deg", 0.0))


fig, axes = plt.subplots(len(problem_idx), 2,
                          figsize=(11, 3.4 * len(problem_idx)),
                          squeeze=False)
for row, j in enumerate(problem_idx):
    c    = curves_app[j]
    d_nm = c["d_hat"] / conv_L
    k    = sel_train.index(j) if j in sel_train else None

    # Original global (Fig. 2)
    if k is not None:
        A_o, phi_o = _aligned_phase_sim_safe(
            c,
            dict(C1=p_glob["C1"], C2=p_glob["C2"], a0=p_glob["a0"],
                 Q=p_glob["Q"], C3=float(p_glob["C3"][k]),
                 d_offset_hat=float(p_glob["d_off"][k]),
                 phi_offset_deg=float(p_glob["p_off"][k])),
        )
        A_a, phi_a = _aligned_phase_sim_safe(
            c,
            dict(C1=p_glob_A["C1"], C2=p_glob_A["C2"], a0=p_glob_A["a0"],
                 Q=p_glob_A["Q"], C3=float(p_glob_A["C3"][k]),
                 d_offset_hat=float(p_glob_A["d_off"][k]),
                 phi_offset_deg=float(p_glob_A["p_off"][k])),
        )

    axA = axes[row, 0]
    axA.plot(d_nm, c["A_hat"] / conv_L, "ko", ms=2.5, mfc="white",
              mew=0.6, label="experiment")
    if k is not None:
        axA.plot(d_nm, A_o / conv_L, "-", color="tab:green",
                  lw=1.1, label="Fig. 2 global")
        axA.plot(d_nm, A_a / conv_L, "-", color="tab:blue",
                  lw=1.4, label="Step A re-seeded")
    axA.set_ylabel("Amplitude (nm)")
    axA.set_title(f"drive = {drive[j]:.2f} nm", fontsize=9)
    axA.legend(frameon=False, fontsize=8)
    axA.set_xlabel("Tip-sample distance (nm)")

    axP = axes[row, 1]
    axP.plot(d_nm, c["phi_deg"], "ko", ms=2.5, mfc="white",
              mew=0.6, label="experiment")
    if k is not None:
        axP.plot(d_nm, phi_o, "-", color="tab:green",
                  lw=1.1, label="Fig. 2 global")
        axP.plot(d_nm, phi_a, "-", color="tab:blue",
                  lw=1.4, label="Step A re-seeded")
    axP.set_ylabel("Phase (deg)")
    axP.set_title(f"drive = {drive[j]:.2f} nm — phase", fontsize=9)
    axP.legend(frameon=False, fontsize=8)
    axP.set_xlabel("Tip-sample distance (nm)")
fig.tight_layout(); plt.show()


# Single-number branch diagnostic for Step A
step_a_status = {}
print("\n  drive   delta_phi_exp   delta_phi_simA   status")
for j in problem_idx:
    c = curves_app[j]
    if j not in sel_train:
        step_a_status[j] = "skip"
        continue
    k = sel_train.index(j)
    _, phi_a = _aligned_phase_sim_safe(
        c,
        dict(C1=p_glob_A["C1"], C2=p_glob_A["C2"], a0=p_glob_A["a0"],
             Q=p_glob_A["Q"], C3=float(p_glob_A["C3"][k]),
             d_offset_hat=float(p_glob_A["d_off"][k]),
             phi_offset_deg=float(p_glob_A["p_off"][k])),
    )
    far_mask  = c["d_hat"] >= np.percentile(c["d_hat"], 85)
    near_mask = c["d_hat"] <= np.percentile(c["d_hat"], 20)
    de = float(np.nanmedian(c["phi_deg"][near_mask])
               - np.nanmedian(c["phi_deg"][far_mask]))
    ds = float(np.nanmedian(phi_a[near_mask])
               - np.nanmedian(phi_a[far_mask]))
    status = "OK " if de * ds > 0 else "still flipped"
    step_a_status[j] = status
    print(f"  {drive[j]:5.2f}     {de:+7.2f}        {ds:+7.2f}       {status}")


In [ ]:
# ---------------------------------------------------------------
# Step B: per-curve C2 and a0 (relaxed shared physics).
# Only runs if Step A still flipped the branch.
# ---------------------------------------------------------------
need_step_B = any("still flipped" in str(v) for v in step_a_status.values())
print(f"Need Step B?  {need_step_B}")

if need_step_B:
    n_shared_log = 6
    n_shared_lin = 1
    base = n_shared_log + n_shared_lin

    def _unpackB(x):
        log_C1, log_Q, log_wc, log_glr, log_llr, log_gcont = x[:n_shared_log]
        dom = x[n_shared_log]
        log_C2 = x[base:base+n_sub]
        log_a0 = x[base+n_sub:base+2*n_sub]
        log_C3 = x[base+2*n_sub:base+3*n_sub]
        d_off  = x[base+3*n_sub:base+4*n_sub]
        p_off  = x[base+4*n_sub:base+5*n_sub]
        return dict(C1=np.exp(log_C1), Q=np.exp(log_Q),
                     w_contact=np.exp(log_wc), gamma_lr=np.exp(log_glr),
                     lambda_lr=np.exp(log_llr),
                     gamma_contact=np.exp(log_gcont),
                     domega=dom,
                     C2=np.exp(log_C2), a0=np.exp(log_a0),
                     C3=np.exp(log_C3), d_off=d_off, p_off=p_off)

    x0_B = np.zeros(n_shared_log + n_shared_lin + 5*n_sub)
    x0_B[:n_shared_log] = np.array([
        _safe_log(p_glob_A["C1"]), _safe_log(p_glob_A["Q"]),
        _safe_log(p_glob_A["w_contact"]),
        _safe_log(p_glob_A["gamma_lr"]),
        _safe_log(p_glob_A["lambda_lr"]),
        _safe_log(p_glob_A["gamma_contact"]),
    ])
    x0_B[n_shared_log] = float(p_glob_A["domega"])

    # Seed per-curve C2 / a0 from Section 4.5 refits where available.
    C2_seed_pc = np.full(n_sub, float(p_glob_A["C2"]))
    a0_seed_pc = np.full(n_sub, float(p_glob_A["a0"]))
    for j in problem_idx:
        if j not in sel_train:
            continue
        k = sel_train.index(j)
        r = refits.get(j, None)
        if r is None:
            continue
        C2_seed_pc[k] = float(r.get("C2", p_glob_A["C2"]))
        a0_seed_pc[k] = float(r.get("a0", p_glob_A["a0"]))

    x0_B[base:base+n_sub]            = np.log(np.maximum(C2_seed_pc, 1e-12))
    x0_B[base+n_sub:base+2*n_sub]    = np.log(np.maximum(a0_seed_pc, 1e-12))
    x0_B[base+2*n_sub:base+3*n_sub]  = np.log(np.maximum(p_glob_A["C3"], 1e-12))
    x0_B[base+3*n_sub:base+4*n_sub]  = p_glob_A["d_off"]
    x0_B[base+4*n_sub:base+5*n_sub]  = p_glob_A["p_off"]

    lo_B = x0_B.copy(); hi_B = x0_B.copy()
    lo_B[0] -= 2.0*np.log(10.0); hi_B[0] += 2.0*np.log(10.0)
    lo_B[1] -= 1.0*np.log(10.0); hi_B[1] += 1.0*np.log(10.0)
    lo_B[2] -= 2.0*np.log(10.0); hi_B[2] += 2.0*np.log(10.0)
    lo_B[3] -= 1.5*np.log(10.0); hi_B[3] += 1.5*np.log(10.0)
    lo_B[4]  = np.log(3.0 * conv_L); hi_B[4] = np.log(40.0 * conv_L)
    lo_B[5] -= 2.0*np.log(10.0); hi_B[5] += 2.0*np.log(10.0)
    lo_B[6]  = -0.03; hi_B[6] = +0.03
    lo_B[base:base+n_sub]            -= 1.5*np.log(10.0)
    hi_B[base:base+n_sub]            += 1.5*np.log(10.0)
    lo_B[base+n_sub:base+2*n_sub]    -= 1.5*np.log(10.0)
    hi_B[base+n_sub:base+2*n_sub]    += 1.5*np.log(10.0)
    lo_B[base+2*n_sub:base+3*n_sub]  -= 1.5*np.log(10.0)
    hi_B[base+2*n_sub:base+3*n_sub]  += 1.5*np.log(10.0)
    lo_B[base+3*n_sub:base+4*n_sub]  = -3.0*conv_L
    hi_B[base+3*n_sub:base+4*n_sub]  = +3.0*conv_L
    lo_B[base+4*n_sub:base+5*n_sub]  = -180.0
    hi_B[base+4*n_sub:base+5*n_sub]  = +180.0

    print("\nRunning Step B per-curve (C2, a0) global re-fit ...")
    t0 = time.time()
    res_B = least_squares(
        _residual_global, x0_B, bounds=(lo_B, hi_B),
        args=(_unpackB,),
        method="trf", max_nfev=400, x_scale="jac", loss="soft_l1",
        verbose=2,
    )
    p_glob_B = _unpackB(res_B.x)
    print(f"Step B done in {time.time()-t0:.1f} s,  cost = {res_B.cost:.3e}")
    print(f"  per-curve C2 (training set): "
          f"{np.round(np.asarray(p_glob_B['C2']), 3).tolist()}")
    print(f"  per-curve a0 (training set): "
          f"{np.round(np.asarray(p_glob_B['a0']), 3).tolist()}")

    fig, axes = plt.subplots(len(problem_idx), 2,
                              figsize=(11, 3.4 * len(problem_idx)),
                              squeeze=False)
    for row, j in enumerate(problem_idx):
        c    = curves_app[j]
        d_nm = c["d_hat"] / conv_L
        if j not in sel_train:
            continue
        k = sel_train.index(j)
        A_b, phi_b = _aligned_phase_sim_safe(
            c,
            dict(C1=p_glob_B["C1"], C2=p_glob_B["C2"][k],
                 a0=p_glob_B["a0"][k], Q=p_glob_B["Q"],
                 C3=float(p_glob_B["C3"][k]),
                 d_offset_hat=float(p_glob_B["d_off"][k]),
                 phi_offset_deg=float(p_glob_B["p_off"][k])),
        )

        axA = axes[row, 0]
        axA.plot(d_nm, c["A_hat"] / conv_L, "ko", ms=2.5, mfc="white",
                  mew=0.6, label="experiment")
        axA.plot(d_nm, A_b / conv_L, "-", color="tab:red",
                  lw=1.4, label="Step B per-curve (C2, a0)")
        axA.set_ylabel("Amplitude (nm)")
        axA.set_title(f"drive = {drive[j]:.2f} nm — amplitude", fontsize=9)
        axA.legend(frameon=False, fontsize=8)
        axA.set_xlabel("Tip-sample distance (nm)")

        axP = axes[row, 1]
        axP.plot(d_nm, c["phi_deg"], "ko", ms=2.5, mfc="white",
                  mew=0.6, label="experiment")
        axP.plot(d_nm, phi_b, "-", color="tab:red",
                  lw=1.4, label="Step B per-curve (C2, a0)")
        axP.set_ylabel("Phase (deg)")
        axP.set_title(f"drive = {drive[j]:.2f} nm — phase", fontsize=9)
        axP.legend(frameon=False, fontsize=8)
        axP.set_xlabel("Tip-sample distance (nm)")
    fig.tight_layout(); plt.show()
else:
    print("Step A already unflipped the branch -- Step B not needed.")


## 4.7  Branch-direction penalty + branch-matched initial conditions

Section 4.6's re-seeded global fit was still amplitude-dominated
(`w_phi = 0.03` vs `w_amp = 1.0`) and started the simulator from the
default `phi_meas_init_deg = 90°`, which on the bifurcation diagram
sits right on the basin boundary at 11.37 nm.  As a result, the
optimizer keeps converging to the repulsive basin: every "wrong"
branch is masked by a perfectly acceptable amplitude fit and the
33× phase down-weighting can't penalise it.

Two knobs available on this FD simulator fix this:

1. **Raise the phase weight to 0.5** so the loss is genuinely
   amplitude + phase balanced.
2. **Add a hard branch-direction penalty.** If `sign(Δφ_near − Δφ_far)`
   in the simulation does not match the experiment, contribute a flat
   penalty per pixel to that curve's residual.  Now even a "perfect"
   amplitude fit on the wrong branch is heavily penalised.
(The third knob — branch-matched initial demodulator phase — is
not available on `simulate_curve_rk45_extended_python`; its lock-in
always rings up from `z = v = 0` and sweeps far → near.  When fixes
1 + 2 alone cannot reach the right basin, the next move is to
relax shared physics with per-curve `(C2, a0)` — Step B in
Section 4.6 — which carries the same effect by adjusting the
bifurcation diagram per curve.)


In [ ]:
# ---------------------------------------------------------------
# Step C: branch-direction penalty in the loss
# (Fix 3 — branch-matched initial demodulator phase — is not
#  available on `simulate_curve_rk45_extended_python`; its lock-in
#  rings up from z = v = 0 every time, sweeping far -> near.  The
#  basin is therefore selected entirely by the force-model
#  parameters and the loss must do the work.)
# ---------------------------------------------------------------
import copy
from scipy.optimize import least_squares

# Loss weights for Step C
W_AMP_C  = 1.0
W_PHI_C  = 0.5       # was 0.03; now genuinely amp+phase balanced
W_BRANCH = 5.0       # flat penalty per pixel when branch direction is wrong


# Per-curve experimental branch direction (sign of Δφ near − Δφ far).
def _branch_sign_exp(c, near_frac=0.20, far_frac=0.85):
    near_mask = c["d_hat"] <= np.percentile(c["d_hat"], 100 * near_frac)
    far_mask  = c["d_hat"] >= np.percentile(c["d_hat"], 100 * far_frac)
    delta = float(np.nanmedian(c["phi_deg"][near_mask])
                  - np.nanmedian(c["phi_deg"][far_mask]))
    return 1.0 if delta > 0 else -1.0      # +1 = attractive, -1 = repulsive


branch_sign_exp_per_curve = np.array(
    [_branch_sign_exp(c) for c in curves_train], dtype=float
)
print("experimental branch direction per training curve:")
for k, (j_tr, s) in enumerate(zip(sel_train, branch_sign_exp_per_curve)):
    print(f"  k={k}  drive={drive[j_tr]:.2f} nm   branch = "
          f"{'attractive (+1)' if s > 0 else 'repulsive (-1)'}")

# Per-curve rk45 options remain identical to the global rk45_fit_options
# (no phi_meas_init_deg injection).  We keep the list shape so the
# downstream residual + plotting code can still iterate per curve.
rk45_fit_options_per_curve = [dict(rk45_fit_options) for _ in curves_train]
print(f"\nUsing identical rk45_fit_options per curve (n = "
      f"{len(rk45_fit_options_per_curve)})")


In [ ]:
# ---------------------------------------------------------------
# Residual: amplitude + phase + branch-direction penalty.
# Works for both shared and per-curve (C2, a0) parameterisations.
# ---------------------------------------------------------------
def _branch_sign_sim(d_hat, phi_use, near_frac=0.20, far_frac=0.85):
    near_mask = d_hat <= np.percentile(d_hat, 100 * near_frac)
    far_mask  = d_hat >= np.percentile(d_hat, 100 * far_frac)
    delta = float(np.nanmedian(phi_use[near_mask])
                  - np.nanmedian(phi_use[far_mask]))
    return 1.0 if delta > 0 else -1.0


def _residual_global_branch(x, _unpack):
    """Same shape as _residual_global but with W_PHI_C phase weight
    and a branch-direction penalty.  Resolves per-curve vs scalar
    (C2, a0) automatically."""
    try:
        p = _unpack(x)
    except Exception:
        return np.full(2 * n_sub * 60, 1e3)
    omega_d = 1.0 + p["domega"]
    res = []
    for k, c in enumerate(curves_train):
        try:
            A_sim, phi_sim, *_ = simulate_curve_rk45_extended_python(
                c["d_hat"],
                C1=float(p["C1"]),
                C2=_resolve_per_curve(p["C2"], k),
                a0=_resolve_per_curve(p["a0"], k),
                Q=float(p["Q"]),
                C3=_resolve_per_curve(p["C3"], k),
                omega_drive_hat=omega_d, omega_ref_hat=1.0,
                d_offset_hat=_resolve_per_curve(p["d_off"], k),
                w_contact=float(p["w_contact"]),
                gamma_lr=float(p["gamma_lr"]),
                lambda_lr=float(p["lambda_lr"]),
                gamma_contact=float(p["gamma_contact"]),
                **rk45_fit_options,
            )
            phi_use = transform_phase_deg(phi_sim, mode=PHASE_MODE)
            phi_off = estimate_phase_offset_far_field(
                phi_use, c["phi_deg"], c["d_hat"],
                frac=0.15, period_deg=360.0,
            )
            phi_use = phi_use + phi_off + _resolve_per_curve(p["p_off"], k)

            r_A   = W_AMP_C * (A_sim - c["A_hat"]) / max(c["A0_hat"], 1e-9)
            r_phi = W_PHI_C * (
                ((phi_use - c["phi_deg"] + 180.0) % 360.0 - 180.0) / 90.0
            )
            s_sim = _branch_sign_sim(c["d_hat"], phi_use)
            r_br  = (W_BRANCH if s_sim != branch_sign_exp_per_curve[k]
                     else 0.0) * np.ones_like(c["d_hat"])

            res.append(np.nan_to_num(r_A,   nan=10.0, posinf=10.0, neginf=-10.0))
            res.append(np.nan_to_num(r_phi, nan=10.0, posinf=10.0, neginf=-10.0))
            res.append(r_br)
        except Exception:
            res.append(np.full_like(c["d_hat"], 10.0))
            res.append(np.full_like(c["d_hat"], 10.0))
            res.append(np.full_like(c["d_hat"], W_BRANCH))
    return np.concatenate(res)


print("Built Step C residual with:")
print(f"  W_AMP    = {W_AMP_C}")
print(f"  W_PHI    = {W_PHI_C}   (was 0.03)")
print(f"  W_BRANCH = {W_BRANCH}   per-pixel penalty for wrong branch")


In [ ]:
# ---------------------------------------------------------------
# Run Step C global re-fit with the branch-direction loss.
# Use the per-curve (C2, a0) parameterisation from Step B if it ran,
# else promote Step A.
# ---------------------------------------------------------------
seed_from_B = (need_step_B and "p_glob_B" in globals()
               and isinstance(p_glob_B.get("C2", None), np.ndarray))
print(f"Seeding Step C from {'Step B (per-curve)' if seed_from_B else 'Step A (shared)'}")

if seed_from_B:
    p_seed = p_glob_B
    unpack_C = _unpackB
    x0_C = res_B.x.copy()
    lo_C = lo_B.copy(); hi_C = hi_B.copy()
else:
    p_seed = p_glob_A

    def _unpack_C_make(n_sub):
        n_shared_log = 6
        n_shared_lin = 1
        base = n_shared_log + n_shared_lin
        def _unpack_C(x):
            log_C1, log_Q, log_wc, log_glr, log_llr, log_gcont = x[:n_shared_log]
            dom = x[n_shared_log]
            log_C2 = x[base:base+n_sub]
            log_a0 = x[base+n_sub:base+2*n_sub]
            log_C3 = x[base+2*n_sub:base+3*n_sub]
            d_off  = x[base+3*n_sub:base+4*n_sub]
            p_off  = x[base+4*n_sub:base+5*n_sub]
            return dict(C1=np.exp(log_C1), Q=np.exp(log_Q),
                         w_contact=np.exp(log_wc), gamma_lr=np.exp(log_glr),
                         lambda_lr=np.exp(log_llr),
                         gamma_contact=np.exp(log_gcont),
                         domega=dom,
                         C2=np.exp(log_C2), a0=np.exp(log_a0),
                         C3=np.exp(log_C3), d_off=d_off, p_off=p_off)
        return _unpack_C

    unpack_C = _unpack_C_make(n_sub)

    n_shared_log = 6; n_shared_lin = 1; base = n_shared_log + n_shared_lin
    x0_C = np.zeros(n_shared_log + n_shared_lin + 5*n_sub)
    x0_C[:n_shared_log] = np.array([
        _safe_log(p_seed["C1"]), _safe_log(p_seed["Q"]),
        _safe_log(p_seed["w_contact"]),
        _safe_log(p_seed["gamma_lr"]),
        _safe_log(p_seed["lambda_lr"]),
        _safe_log(p_seed["gamma_contact"]),
    ])
    x0_C[n_shared_log] = float(p_seed["domega"])
    x0_C[base:base+n_sub]            = np.log(np.full(n_sub, float(p_seed["C2"])))
    x0_C[base+n_sub:base+2*n_sub]    = np.log(np.full(n_sub, float(p_seed["a0"])))
    x0_C[base+2*n_sub:base+3*n_sub]  = np.log(np.maximum(p_seed["C3"], 1e-12))
    x0_C[base+3*n_sub:base+4*n_sub]  = p_seed["d_off"]
    x0_C[base+4*n_sub:base+5*n_sub]  = p_seed["p_off"]

    lo_C = x0_C.copy(); hi_C = x0_C.copy()
    lo_C[0] -= 2.0*np.log(10.0); hi_C[0] += 2.0*np.log(10.0)
    lo_C[1] -= 1.0*np.log(10.0); hi_C[1] += 1.0*np.log(10.0)
    lo_C[2] -= 2.0*np.log(10.0); hi_C[2] += 2.0*np.log(10.0)
    lo_C[3] -= 1.5*np.log(10.0); hi_C[3] += 1.5*np.log(10.0)
    lo_C[4]  = np.log(3.0 * conv_L); hi_C[4] = np.log(40.0 * conv_L)
    lo_C[5] -= 2.0*np.log(10.0); hi_C[5] += 2.0*np.log(10.0)
    lo_C[6]  = -0.03; hi_C[6] = +0.03
    lo_C[base:base+n_sub]            -= 1.5*np.log(10.0)
    hi_C[base:base+n_sub]            += 1.5*np.log(10.0)
    lo_C[base+n_sub:base+2*n_sub]    -= 1.5*np.log(10.0)
    hi_C[base+n_sub:base+2*n_sub]    += 1.5*np.log(10.0)
    lo_C[base+2*n_sub:base+3*n_sub]  -= 1.5*np.log(10.0)
    hi_C[base+2*n_sub:base+3*n_sub]  += 1.5*np.log(10.0)
    lo_C[base+3*n_sub:base+4*n_sub]  = -3.0*conv_L
    hi_C[base+3*n_sub:base+4*n_sub]  = +3.0*conv_L
    lo_C[base+4*n_sub:base+5*n_sub]  = -180.0
    hi_C[base+4*n_sub:base+5*n_sub]  = +180.0


print("\nRunning Step C global re-fit (branch-direction penalty) ...")
t0 = time.time()
res_C = least_squares(
    _residual_global_branch, x0_C, bounds=(lo_C, hi_C),
    args=(unpack_C,),
    method="trf", max_nfev=400, x_scale="jac", loss="soft_l1",
    verbose=2,
)
p_glob_C = unpack_C(res_C.x)
print(f"Step C done in {time.time()-t0:.1f} s,  cost = {res_C.cost:.3e}")

# Diagnose: did Step C unflip every problem drive?
print("\n  drive   delta_phi_exp   delta_phi_simC   status")
for j in problem_idx:
    c = curves_app[j]
    if j not in sel_train:
        continue
    k = sel_train.index(j)
    A_c, phi_c_raw, *_ = simulate_curve_rk45_extended_python(
        c["d_hat"],
        C1=float(p_glob_C["C1"]),
        C2=_resolve_per_curve(p_glob_C["C2"], k),
        a0=_resolve_per_curve(p_glob_C["a0"], k),
        Q=float(p_glob_C["Q"]),
        C3=_resolve_per_curve(p_glob_C["C3"], k),
        omega_drive_hat=1.0 + p_glob_C["domega"], omega_ref_hat=1.0,
        d_offset_hat=_resolve_per_curve(p_glob_C["d_off"], k),
        w_contact=float(p_glob_C["w_contact"]),
        gamma_lr=float(p_glob_C["gamma_lr"]),
        lambda_lr=float(p_glob_C["lambda_lr"]),
        gamma_contact=float(p_glob_C["gamma_contact"]),
        **rk45_fit_options,
    )
    phi_use = transform_phase_deg(phi_c_raw, mode=PHASE_MODE)
    phi_off = estimate_phase_offset_far_field(
        phi_use, c["phi_deg"], c["d_hat"], frac=0.15, period_deg=360.0,
    )
    phi_use = phi_use + phi_off + _resolve_per_curve(p_glob_C["p_off"], k)
    far_mask  = c["d_hat"] >= np.percentile(c["d_hat"], 85)
    near_mask = c["d_hat"] <= np.percentile(c["d_hat"], 20)
    de = float(np.nanmedian(c["phi_deg"][near_mask])
               - np.nanmedian(c["phi_deg"][far_mask]))
    ds = float(np.nanmedian(phi_use[near_mask])
               - np.nanmedian(phi_use[far_mask]))
    status = "OK " if de * ds > 0 else "still flipped"
    print(f"  {drive[j]:5.2f}     {de:+7.2f}        {ds:+7.2f}       {status}")


In [ ]:
# ---------------------------------------------------------------
# Compare Fig. 2 global, Step A, Step C at each problem drive.
# ---------------------------------------------------------------
fig, axes = plt.subplots(len(problem_idx), 2,
                          figsize=(11, 3.4 * len(problem_idx)),
                          squeeze=False)

def _simulate_p(c, p_dict, opts):
    A_sim, phi_sim, *_ = simulate_curve_rk45_extended_python(
        c["d_hat"],
        C1=float(p_dict["C1"]), C2=float(p_dict["C2"]),
        a0=float(p_dict["a0"]),  Q=float(p_dict["Q"]),
        C3=float(p_dict["C3"]),
        omega_drive_hat=1.0, omega_ref_hat=1.0,
        d_offset_hat=float(p_dict["d_offset_hat"]),
        w_contact=fixed_extra["w_contact"],
        gamma_lr=fixed_extra["gamma_lr"],
        lambda_lr=fixed_extra["lambda_lr"],
        gamma_contact=fixed_extra["gamma_contact"],
        **opts,
    )
    phi_use = transform_phase_deg(phi_sim, mode=PHASE_MODE)
    phi_off = estimate_phase_offset_far_field(
        phi_use, c["phi_deg"], c["d_hat"], frac=0.15, period_deg=360.0,
    )
    return A_sim, phi_use + phi_off + float(p_dict.get("phi_offset_deg", 0.0))


for row, j in enumerate(problem_idx):
    c    = curves_app[j]
    d_nm = c["d_hat"] / conv_L
    if j not in sel_train:
        continue
    k = sel_train.index(j)

    # Fig. 2 result
    A_o, phi_o = _simulate_p(
        c,
        dict(C1=p_glob["C1"], C2=p_glob["C2"], a0=p_glob["a0"],
             Q=p_glob["Q"], C3=float(p_glob["C3"][k]),
             d_offset_hat=float(p_glob["d_off"][k]),
             phi_offset_deg=float(p_glob["p_off"][k])),
        rk45_plot_options,
    )
    # Step A
    A_a, phi_a = _simulate_p(
        c,
        dict(C1=p_glob_A["C1"], C2=p_glob_A["C2"], a0=p_glob_A["a0"],
             Q=p_glob_A["Q"], C3=float(p_glob_A["C3"][k]),
             d_offset_hat=float(p_glob_A["d_off"][k]),
             phi_offset_deg=float(p_glob_A["p_off"][k])),
        rk45_plot_options,
    )
    # Step C (per-curve C2, a0, branch-direction loss)
    A_c, phi_c = _simulate_p(
        c,
        dict(C1=p_glob_C["C1"], C2=_resolve_per_curve(p_glob_C["C2"], k),
             a0=_resolve_per_curve(p_glob_C["a0"], k),
             Q=p_glob_C["Q"], C3=float(p_glob_C["C3"][k]),
             d_offset_hat=float(p_glob_C["d_off"][k]),
             phi_offset_deg=float(p_glob_C["p_off"][k])),
        rk45_plot_options,
    )

    axA = axes[row, 0]
    axA.plot(d_nm, c["A_hat"] / conv_L, "ko", ms=2.5, mfc="white",
              mew=0.6, label="experiment")
    axA.plot(d_nm, A_o / conv_L, "-", color="tab:green",
              lw=1.0, label="Fig. 2 global")
    axA.plot(d_nm, A_a / conv_L, "-", color="tab:blue",
              lw=1.0, label="Step A re-seeded")
    axA.plot(d_nm, A_c / conv_L, "-", color="tab:red",
              lw=1.6, label="Step C branch-aware")
    axA.set_ylabel("Amplitude (nm)")
    axA.set_title(f"drive = {drive[j]:.2f} nm", fontsize=9)
    axA.legend(frameon=False, fontsize=8)
    axA.set_xlabel("Tip-sample distance (nm)")

    axP = axes[row, 1]
    axP.plot(d_nm, c["phi_deg"], "ko", ms=2.5, mfc="white",
              mew=0.6, label="experiment")
    axP.plot(d_nm, phi_o, "-", color="tab:green",
              lw=1.0, label="Fig. 2 global")
    axP.plot(d_nm, phi_a, "-", color="tab:blue",
              lw=1.0, label="Step A re-seeded")
    axP.plot(d_nm, phi_c, "-", color="tab:red",
              lw=1.6, label="Step C branch-aware")
    axP.set_ylabel("Phase (deg)")
    axP.set_title(f"drive = {drive[j]:.2f} nm - phase", fontsize=9)
    axP.legend(frameon=False, fontsize=8)
    axP.set_xlabel("Tip-sample distance (nm)")

fig.tight_layout(); plt.show()
print("\nStep C parameters in `p_glob_C`.  To use them for Fig. 2 of")
print("the paper, replace p_glob with p_glob_C in the figure cell.")


## 4.8  All-curve diagnostic + per-curve w_contact (Step D)

Section 4.7 reduced the 11.37 nm phase deviation from −48° to −17° but did
not actually unflip it; the Step C cost of 313 sits inside the branch
penalty.  Two follow-ups, in order:

**Diagnostic.**  Print Δφ_exp vs Δφ_sim for *every* training drive (not
just the one in `problem_idx`).  If only 11.37 nm is flipped, the cost
is dominated by one curve worth ≈250.  If three curves are flipped
(11.37 + 8.35 + 5.34, say), the cost ≈750 is split across them and the
optimizer was juggling multiple branch errors at once.

**Step D — per-curve w_contact.**  `w_contact` sets the smoothing
width of the contact transition and therefore directly controls where
the bistability boundary sits on the `(drive, distance)` plane.
Making it per-curve, on top of the already per-curve `(C2, a0)`,
admits that the cantilever's effective contact width differs at
intermediate drive — honest physics for an instrument crossing the
snap-in regime.  The bounds on per-curve `C2` and `a0` are also
relaxed (±3·log10 instead of ±1.5·log10) so the optimizer has the
room it didn't have in Step B.


In [ ]:
# ---------------------------------------------------------------
# D1: All-curve diagnostic for the Step C fit.
# ---------------------------------------------------------------
def _eval_step_C(c, k):
    A_sim, phi_sim, *_ = simulate_curve_rk45_extended_python(
        c["d_hat"],
        C1=float(p_glob_C["C1"]),
        C2=_resolve_per_curve(p_glob_C["C2"], k),
        a0=_resolve_per_curve(p_glob_C["a0"], k),
        Q=float(p_glob_C["Q"]),
        C3=_resolve_per_curve(p_glob_C["C3"], k),
        omega_drive_hat=1.0 + p_glob_C["domega"], omega_ref_hat=1.0,
        d_offset_hat=_resolve_per_curve(p_glob_C["d_off"], k),
        w_contact=float(p_glob_C["w_contact"]),
        gamma_lr=float(p_glob_C["gamma_lr"]),
        lambda_lr=float(p_glob_C["lambda_lr"]),
        gamma_contact=float(p_glob_C["gamma_contact"]),
        **rk45_fit_options,
    )
    phi_use = transform_phase_deg(phi_sim, mode=PHASE_MODE)
    phi_off = estimate_phase_offset_far_field(
        phi_use, c["phi_deg"], c["d_hat"], frac=0.15, period_deg=360.0,
    )
    phi_use = phi_use + phi_off + _resolve_per_curve(p_glob_C["p_off"], k)

    near_mask = c["d_hat"] <= np.percentile(c["d_hat"], 20)
    far_mask  = c["d_hat"] >= np.percentile(c["d_hat"], 85)
    de = float(np.nanmedian(c["phi_deg"][near_mask])
               - np.nanmedian(c["phi_deg"][far_mask]))
    ds = float(np.nanmedian(phi_use[near_mask])
               - np.nanmedian(phi_use[far_mask]))
    A_sse = float(np.nansum(((A_sim - c["A_hat"]) / max(c["A0_hat"], 1e-9)) ** 2))
    return de, ds, A_sse


print("Step C — branch status for all training drives")
print(f"{'k':>2}  {'drive':>6}  {'exp_branch':>10}  {'Δφ_exp':>8}  "
      f"{'Δφ_sim':>8}  {'status':>14}  {'C2':>8}  {'a0':>8}  {'A_SSE':>8}")
print("-" * 86)
n_flipped = 0
for k, c in enumerate(curves_train):
    de, ds, A_sse = _eval_step_C(c, k)
    exp_br = "attractive" if branch_sign_exp_per_curve[k] > 0 else "repulsive "
    status = "OK " if de * ds > 0 else "FLIPPED"
    if status == "FLIPPED":
        n_flipped += 1
    print(f"{k:>2}  {drive[sel_train[k]]:>6.2f}  {exp_br:>10}  "
          f"{de:>+8.2f}  {ds:>+8.2f}  {status:>14}  "
          f"{_resolve_per_curve(p_glob_C['C2'], k):>8.3f}  "
          f"{_resolve_per_curve(p_glob_C['a0'], k):>8.3f}  {A_sse:>8.3f}")

penalty_per_pixel = float(W_BRANCH)
mean_n_pix = float(np.mean([len(c["d_hat"]) for c in curves_train]))
expected_branch_cost = n_flipped * penalty_per_pixel ** 2 * mean_n_pix / 2.0
print()
print(f"Flipped curves : {n_flipped} of {len(curves_train)}")
print(f"Expected branch contribution to cost (half-SSR):  "
      f"≈ {expected_branch_cost:.0f}     (actual cost = {res_C.cost:.0f})")


In [ ]:
# ---------------------------------------------------------------
# D2: Step D — per-curve (C2, a0, w_contact) with widened bounds.
# Everything else (Q, gamma_lr, lambda_lr, gamma_contact, domega)
# stays shared.
# ---------------------------------------------------------------
import time as _time

n_shared_log_D = 5     # log_C1, log_Q, log_glr, log_llr, log_gcont
n_shared_lin_D = 1     # domega
baseD = n_shared_log_D + n_shared_lin_D


def _unpackD(x):
    log_C1, log_Q, log_glr, log_llr, log_gcont = x[:n_shared_log_D]
    dom = x[n_shared_log_D]
    log_C2 = x[baseD:baseD+n_sub]
    log_a0 = x[baseD+n_sub:baseD+2*n_sub]
    log_wc = x[baseD+2*n_sub:baseD+3*n_sub]
    log_C3 = x[baseD+3*n_sub:baseD+4*n_sub]
    d_off  = x[baseD+4*n_sub:baseD+5*n_sub]
    p_off  = x[baseD+5*n_sub:baseD+6*n_sub]
    return dict(C1=np.exp(log_C1), Q=np.exp(log_Q),
                gamma_lr=np.exp(log_glr),
                lambda_lr=np.exp(log_llr),
                gamma_contact=np.exp(log_gcont),
                domega=dom,
                C2=np.exp(log_C2), a0=np.exp(log_a0),
                w_contact=np.exp(log_wc),
                C3=np.exp(log_C3), d_off=d_off, p_off=p_off)


# Seed everything from p_glob_C (which already has per-curve C2, a0).
x0_D = np.zeros(n_shared_log_D + n_shared_lin_D + 6 * n_sub)
x0_D[:n_shared_log_D] = np.array([
    _safe_log(p_glob_C["C1"]),
    _safe_log(p_glob_C["Q"]),
    _safe_log(p_glob_C["gamma_lr"]),
    _safe_log(p_glob_C["lambda_lr"]),
    _safe_log(p_glob_C["gamma_contact"]),
])
x0_D[n_shared_log_D] = float(p_glob_C["domega"])

# Per-curve seeds.  C2, a0 already per-curve in p_glob_C.  w_contact was
# shared; broadcast to all curves as starting point.
C2_seed_D = np.array(
    [_resolve_per_curve(p_glob_C["C2"], k) for k in range(n_sub)]
)
a0_seed_D = np.array(
    [_resolve_per_curve(p_glob_C["a0"], k) for k in range(n_sub)]
)
wc_seed_D = np.full(n_sub, float(p_glob_C["w_contact"]))

x0_D[baseD:baseD+n_sub]            = np.log(np.maximum(C2_seed_D, 1e-12))
x0_D[baseD+n_sub:baseD+2*n_sub]    = np.log(np.maximum(a0_seed_D, 1e-12))
x0_D[baseD+2*n_sub:baseD+3*n_sub]  = np.log(np.maximum(wc_seed_D, 1e-12))
x0_D[baseD+3*n_sub:baseD+4*n_sub]  = np.log(np.maximum(p_glob_C["C3"], 1e-12))
x0_D[baseD+4*n_sub:baseD+5*n_sub]  = p_glob_C["d_off"]
x0_D[baseD+5*n_sub:baseD+6*n_sub]  = p_glob_C["p_off"]


# Bounds.  Shared get the same width as Step C.
lo_D = x0_D.copy(); hi_D = x0_D.copy()
lo_D[0] -= 2.0*np.log(10.0); hi_D[0] += 2.0*np.log(10.0)            # log_C1
lo_D[1] -= 1.0*np.log(10.0); hi_D[1] += 1.0*np.log(10.0)            # log_Q
lo_D[2] -= 1.5*np.log(10.0); hi_D[2] += 1.5*np.log(10.0)            # log_glr
lo_D[3]  = np.log(3.0 * conv_L); hi_D[3] = np.log(40.0 * conv_L)    # log_llr
lo_D[4] -= 2.0*np.log(10.0); hi_D[4] += 2.0*np.log(10.0)            # log_gcont
lo_D[5]  = -0.03; hi_D[5] = +0.03                                    # domega

# Per-curve C2, a0 WIDENED to ±3·log10 (was ±1.5)
lo_D[baseD:baseD+n_sub]            -= 3.0*np.log(10.0)
hi_D[baseD:baseD+n_sub]            += 3.0*np.log(10.0)
lo_D[baseD+n_sub:baseD+2*n_sub]    -= 3.0*np.log(10.0)
hi_D[baseD+n_sub:baseD+2*n_sub]    += 3.0*np.log(10.0)

# Per-curve w_contact: ±2·log10 around the shared starting value.
lo_D[baseD+2*n_sub:baseD+3*n_sub]  -= 2.0*np.log(10.0)
hi_D[baseD+2*n_sub:baseD+3*n_sub]  += 2.0*np.log(10.0)

# Per-curve C3 / d_off / p_off — same as Step C.
lo_D[baseD+3*n_sub:baseD+4*n_sub]  -= 1.5*np.log(10.0)
hi_D[baseD+3*n_sub:baseD+4*n_sub]  += 1.5*np.log(10.0)
lo_D[baseD+4*n_sub:baseD+5*n_sub]   = -3.0*conv_L
hi_D[baseD+4*n_sub:baseD+5*n_sub]   = +3.0*conv_L
lo_D[baseD+5*n_sub:baseD+6*n_sub]   = -180.0
hi_D[baseD+5*n_sub:baseD+6*n_sub]   = +180.0


# Reuse the branch-aware residual.  _residual_global_branch already
# uses _resolve_per_curve on every parameter, so it works for both
# scalar and array entries — but w_contact was scalar there.  The
# unified call below passes float(p["w_contact"]) which fails for
# arrays.  We need a tiny wrapper.

def _residual_global_branch_D(x):
    """Same as _residual_global_branch but resolves w_contact per curve."""
    try:
        p = _unpackD(x)
    except Exception:
        return np.full(2 * n_sub * 60, 1e3)
    omega_d = 1.0 + p["domega"]
    res = []
    for k, c in enumerate(curves_train):
        try:
            A_sim, phi_sim, *_ = simulate_curve_rk45_extended_python(
                c["d_hat"],
                C1=float(p["C1"]),
                C2=_resolve_per_curve(p["C2"], k),
                a0=_resolve_per_curve(p["a0"], k),
                Q=float(p["Q"]),
                C3=_resolve_per_curve(p["C3"], k),
                omega_drive_hat=omega_d, omega_ref_hat=1.0,
                d_offset_hat=_resolve_per_curve(p["d_off"], k),
                w_contact=_resolve_per_curve(p["w_contact"], k),
                gamma_lr=float(p["gamma_lr"]),
                lambda_lr=float(p["lambda_lr"]),
                gamma_contact=float(p["gamma_contact"]),
                **rk45_fit_options,
            )
            phi_use = transform_phase_deg(phi_sim, mode=PHASE_MODE)
            phi_off = estimate_phase_offset_far_field(
                phi_use, c["phi_deg"], c["d_hat"],
                frac=0.15, period_deg=360.0,
            )
            phi_use = phi_use + phi_off + _resolve_per_curve(p["p_off"], k)
            r_A   = W_AMP_C * (A_sim - c["A_hat"]) / max(c["A0_hat"], 1e-9)
            r_phi = W_PHI_C * (
                ((phi_use - c["phi_deg"] + 180.0) % 360.0 - 180.0) / 90.0
            )
            s_sim = _branch_sign_sim(c["d_hat"], phi_use)
            r_br  = (W_BRANCH if s_sim != branch_sign_exp_per_curve[k]
                     else 0.0) * np.ones_like(c["d_hat"])
            res.append(np.nan_to_num(r_A,   nan=10.0, posinf=10.0, neginf=-10.0))
            res.append(np.nan_to_num(r_phi, nan=10.0, posinf=10.0, neginf=-10.0))
            res.append(r_br)
        except Exception:
            res.append(np.full_like(c["d_hat"], 10.0))
            res.append(np.full_like(c["d_hat"], 10.0))
            res.append(np.full_like(c["d_hat"], W_BRANCH))
    return np.concatenate(res)


print("\nRunning Step D global re-fit "
      "(per-curve C2/a0/w_contact, widened bounds) ...")
t0 = _time.time()
res_D = least_squares(
    _residual_global_branch_D, x0_D, bounds=(lo_D, hi_D),
    method="trf", max_nfev=500, x_scale="jac", loss="soft_l1",
    verbose=2,
)
p_glob_D = _unpackD(res_D.x)
print(f"Step D done in {_time.time()-t0:.1f} s,  cost = {res_D.cost:.3e}")
print(f"  per-curve C2:        {np.round(p_glob_D['C2'], 3).tolist()}")
print(f"  per-curve a0:        {np.round(p_glob_D['a0'], 3).tolist()}")
print(f"  per-curve w_contact: {np.round(p_glob_D['w_contact'], 3).tolist()}")


In [ ]:
# ---------------------------------------------------------------
# D3: Branch status after Step D + comparison plot vs Step C.
# ---------------------------------------------------------------
def _eval_step_D(c, k):
    A_sim, phi_sim, *_ = simulate_curve_rk45_extended_python(
        c["d_hat"],
        C1=float(p_glob_D["C1"]),
        C2=_resolve_per_curve(p_glob_D["C2"], k),
        a0=_resolve_per_curve(p_glob_D["a0"], k),
        Q=float(p_glob_D["Q"]),
        C3=_resolve_per_curve(p_glob_D["C3"], k),
        omega_drive_hat=1.0 + p_glob_D["domega"], omega_ref_hat=1.0,
        d_offset_hat=_resolve_per_curve(p_glob_D["d_off"], k),
        w_contact=_resolve_per_curve(p_glob_D["w_contact"], k),
        gamma_lr=float(p_glob_D["gamma_lr"]),
        lambda_lr=float(p_glob_D["lambda_lr"]),
        gamma_contact=float(p_glob_D["gamma_contact"]),
        **rk45_fit_options,
    )
    phi_use = transform_phase_deg(phi_sim, mode=PHASE_MODE)
    phi_off = estimate_phase_offset_far_field(
        phi_use, c["phi_deg"], c["d_hat"], frac=0.15, period_deg=360.0,
    )
    phi_use = phi_use + phi_off + _resolve_per_curve(p_glob_D["p_off"], k)

    near_mask = c["d_hat"] <= np.percentile(c["d_hat"], 20)
    far_mask  = c["d_hat"] >= np.percentile(c["d_hat"], 85)
    de = float(np.nanmedian(c["phi_deg"][near_mask])
               - np.nanmedian(c["phi_deg"][far_mask]))
    ds = float(np.nanmedian(phi_use[near_mask])
               - np.nanmedian(phi_use[far_mask]))
    return A_sim, phi_use, de, ds


print("Step D — branch status for all training drives")
print(f"{'k':>2}  {'drive':>6}  {'exp_branch':>10}  "
      f"{'Δφ_exp':>8}  {'Δφ_sim':>8}  {'status':>14}  "
      f"{'C2':>8}  {'a0':>8}  {'w_c':>8}")
print("-" * 92)
n_flipped_D = 0
for k, c in enumerate(curves_train):
    _, _, de, ds = _eval_step_D(c, k)
    exp_br = "attractive" if branch_sign_exp_per_curve[k] > 0 else "repulsive "
    status = "OK " if de * ds > 0 else "FLIPPED"
    if status == "FLIPPED":
        n_flipped_D += 1
    print(f"{k:>2}  {drive[sel_train[k]]:>6.2f}  {exp_br:>10}  "
          f"{de:>+8.2f}  {ds:>+8.2f}  {status:>14}  "
          f"{_resolve_per_curve(p_glob_D['C2'], k):>8.3f}  "
          f"{_resolve_per_curve(p_glob_D['a0'], k):>8.3f}  "
          f"{_resolve_per_curve(p_glob_D['w_contact'], k):>8.3f}")
print()
print(f"Step C flipped: {n_flipped} of {len(curves_train)}")
print(f"Step D flipped: {n_flipped_D} of {len(curves_train)}")


# Plot every training curve, Step C vs Step D
fig, axes = plt.subplots(2, len(curves_train),
                          figsize=(2.5 * len(curves_train), 5),
                          squeeze=False)
for k, c in enumerate(curves_train):
    d_nm = c["d_hat"] / conv_L
    A_c_arr, phi_c_arr, *_ = simulate_curve_rk45_extended_python(
        c["d_hat"],
        C1=float(p_glob_C["C1"]),
        C2=_resolve_per_curve(p_glob_C["C2"], k),
        a0=_resolve_per_curve(p_glob_C["a0"], k),
        Q=float(p_glob_C["Q"]),
        C3=_resolve_per_curve(p_glob_C["C3"], k),
        omega_drive_hat=1.0 + p_glob_C["domega"], omega_ref_hat=1.0,
        d_offset_hat=_resolve_per_curve(p_glob_C["d_off"], k),
        w_contact=float(p_glob_C["w_contact"]),
        gamma_lr=float(p_glob_C["gamma_lr"]),
        lambda_lr=float(p_glob_C["lambda_lr"]),
        gamma_contact=float(p_glob_C["gamma_contact"]),
        **rk45_fit_options,
    )
    phi_c = transform_phase_deg(phi_c_arr, mode=PHASE_MODE)
    phi_c = phi_c + estimate_phase_offset_far_field(
        phi_c, c["phi_deg"], c["d_hat"], frac=0.15, period_deg=360.0,
    ) + _resolve_per_curve(p_glob_C["p_off"], k)

    A_d_arr, phi_d_arr, de, ds = _eval_step_D(c, k)

    axA = axes[0, k]
    axA.plot(d_nm, c["A_hat"] / conv_L, "ko", ms=2, mfc="white", mew=0.5)
    axA.plot(d_nm, A_c_arr / conv_L, "-", color="tab:red",  lw=1.0, alpha=0.7,
              label="C" if k == 0 else None)
    axA.plot(d_nm, A_d_arr / conv_L, "-", color="tab:purple", lw=1.4,
              label="D" if k == 0 else None)
    axA.set_title(f"drive = {drive[sel_train[k]]:.2f} nm", fontsize=8)
    axA.tick_params(labelsize=7)
    if k == 0:
        axA.legend(fontsize=7, frameon=False)
        axA.set_ylabel("A (nm)")

    axP = axes[1, k]
    axP.plot(d_nm, c["phi_deg"], "ko", ms=2, mfc="white", mew=0.5)
    axP.plot(d_nm, phi_c, "-", color="tab:red",  lw=1.0, alpha=0.7)
    axP.plot(d_nm, phi_d_arr, "-", color="tab:purple", lw=1.4)
    axP.set_xlabel("d (nm)", fontsize=8)
    axP.tick_params(labelsize=7)
    if k == 0:
        axP.set_ylabel("phi (deg)")
fig.suptitle("Step C (red) vs Step D (purple) on every training drive",
              fontsize=10)
fig.tight_layout(rect=[0, 0, 1, 0.97])
plt.show()
print("\nIf Step D unflipped 11.37 nm (and any other previously-flipped")
print("curves), use p_glob_D for the manuscript figure.")


## 4.9  Checkpoint — save / load every fit result

All the fits above (individual round-1, individual round-2, global,
Section 4.5 branch-aware refit, Section 4.6 Step A / Step B, Section
4.7 Step C, Section 4.8 Step D) take a long time to re-run on a cold
kernel.  Below we dump everything to a single joblib bundle and
provide a matching loader, so the publication-figure cells further
down can be run cold without re-fitting anything.

* **Cell S1** saves into
  `output/fd_calibration_Tap300_v2_results.joblib`.  Stages that were
  never run are skipped automatically (they just don't appear in the
  bundle).
* **Cell S2** is the loader.  If the bundle exists, it rebinds every
  name (`ind_fits`, `ind_fits_v4`, `p_glob`, `refits`, `p_glob_A`,
  `p_glob_B`, `p_glob_C`, `p_glob_D`, plus `sel_train`, `sel_eval`,
  `curves_app_sub`, `branch_sign_exp_per_curve`, `problem_idx`,
  `fixed_extra`, `rk45_fit_options`, `rk45_plot_options`, `PHASE_MODE`,
  `conv_L`, etc.) into the global namespace, so the publication figure
  cells below can be executed directly.


In [ ]:
# # ---------------------------------------------------------------
# # S1: Save every fit result + config to a joblib bundle.
# # ---------------------------------------------------------------
# from joblib import dump as _joblib_dump
# import os as _os

# _save_dir = _os.path.join(PROJECT_ROOT if "PROJECT_ROOT" in globals() else ".", "output")
# _os.makedirs(_save_dir, exist_ok=True)
# SAVE_PATH = _os.path.join(_save_dir, "fd_calibration_Tap300_v2_results.joblib")

# # Convert anything keyed on numpy ints (e.g. refits keyed on j as np.int)
# # to plain int keys so loading is independent of numpy version.
# def _to_plain(obj):
#     if isinstance(obj, dict):
#         return {(int(k) if hasattr(k, "item") else k): _to_plain(v)
#                 for k, v in obj.items()}
#     return obj


# def _maybe(name):
#     return globals().get(name, None)


# bundle = {
#     # --- Raw experimental inputs ---
#     "drive":        _maybe("drive"),
#     "conv_L":       _maybe("conv_L"),
#     "curves_app":   _maybe("curves_app"),
#     "curves_ret":   _maybe("curves_ret"),
#     "curves_app_sub": _maybe("curves_app_sub"),
#     "curves_train": _maybe("curves_train"),
#     "curves_eval":  _maybe("curves_eval"),

#     # --- Selection ---
#     "sel_train": _maybe("sel_train"),
#     "sel_eval":  _maybe("sel_eval"),
#     "good_idx":  _maybe("good_idx"),

#     # --- Solver knobs / shared ---
#     "rk45_fit_options":  _maybe("rk45_fit_options"),
#     "rk45_plot_options": _maybe("rk45_plot_options"),
#     "fixed_extra":       _maybe("fixed_extra"),
#     "PHASE_MODE":        _maybe("PHASE_MODE"),
#     "Q_TARGET":          _maybe("Q_TARGET"),
#     "K_N_PER_M":         _maybe("K_N_PER_M"),
#     "F0_HZ":             _maybe("F0_HZ"),

#     # --- Workflow knobs ---
#     "N_TRAIN_CURVES":     _maybe("N_TRAIN_CURVES"),
#     "N_HELDOUT_CURVES":   _maybe("N_HELDOUT_CURVES"),
#     "LOW_A_CUTOFF_FRAC":  _maybe("LOW_A_CUTOFF_FRAC"),
#     "DRIVE_MIN_NM":       _maybe("DRIVE_MIN_NM"),

#     # --- Stage outputs ---
#     "ind_fits":     _maybe("ind_fits"),     # round-1 individual
#     "ind_fits_v4":  _maybe("ind_fits_v4"),  # round-2 individual (shared physics locked)
#     "p_glob":       _maybe("p_glob"),

#     # --- Section 4.5 branch-aware refit ---
#     "refits":              _to_plain(_maybe("refits")) if _maybe("refits") else None,
#     "problem_idx":         _maybe("problem_idx"),
#     "PROBLEM_DRIVES_NM":   _maybe("PROBLEM_DRIVES_NM"),

#     # --- Section 4.6 ---
#     "ind_fits_patched": _maybe("ind_fits_patched"),
#     "p_glob_A":         _maybe("p_glob_A"),
#     "p_glob_B":         _maybe("p_glob_B"),
#     "need_step_B":      _maybe("need_step_B"),
#     "step_a_status":    _to_plain(_maybe("step_a_status")) if _maybe("step_a_status") else None,

#     # --- Section 4.7 / Step C ---
#     "branch_sign_exp_per_curve": _maybe("branch_sign_exp_per_curve"),
#     "W_AMP_C":  _maybe("W_AMP_C"),
#     "W_PHI_C":  _maybe("W_PHI_C"),
#     "W_BRANCH": _maybe("W_BRANCH"),
#     "p_glob_C": _maybe("p_glob_C"),

#     # --- Section 4.8 / Step D ---
#     "p_glob_D": _maybe("p_glob_D"),
#     "n_flipped":   _maybe("n_flipped"),
#     "n_flipped_D": _maybe("n_flipped_D"),

#     # --- Lightweight residual costs (so we can print without re-running) ---
#     "cost_glob":   getattr(_maybe("res_glob"), "cost", None) if _maybe("res_glob") else None,
#     "cost_A":      getattr(_maybe("res_A"),    "cost", None) if _maybe("res_A")    else None,
#     "cost_B":      getattr(_maybe("res_B"),    "cost", None) if _maybe("res_B")    else None,
#     "cost_C":      getattr(_maybe("res_C"),    "cost", None) if _maybe("res_C")    else None,
#     "cost_D":      getattr(_maybe("res_D"),    "cost", None) if _maybe("res_D")    else None,
# }

# # Count what we actually have so the user knows which stages are in the bundle.
# present = [k for k, v in bundle.items() if v is not None]
# print(f"Saving {len(present)} keys to {SAVE_PATH}")
# print(f"  stages present: ind_fits = {'ind_fits' in present}, "
#       f"ind_fits_v4 = {'ind_fits_v4' in present}, "
#       f"p_glob = {'p_glob' in present}, "
#       f"refits = {'refits' in present}, "
#       f"A = {'p_glob_A' in present}, "
#       f"B = {'p_glob_B' in present}, "
#       f"C = {'p_glob_C' in present}, "
#       f"D = {'p_glob_D' in present}")

# _joblib_dump(bundle, SAVE_PATH, compress=3)
# print(f"Saved.  File size ≈ "
#       f"{_os.path.getsize(SAVE_PATH) / (1024**2):.2f} MB")


In [ ]:
# # ---------------------------------------------------------------
# # S2: Load every fit result + config from the saved bundle.
# # Run this if you opened the notebook cold and want to skip the
# # heavy fits to go straight to the publication figures.
# # ---------------------------------------------------------------
# from joblib import load as _joblib_load
# import os as _os

# _save_dir = _os.path.join(PROJECT_ROOT if "PROJECT_ROOT" in globals() else ".", "output")
# SAVE_PATH = _os.path.join(_save_dir, "fd_calibration_Tap300_v2_results.joblib")

# if not _os.path.exists(SAVE_PATH):
#     raise FileNotFoundError(
#         f"No saved bundle at {SAVE_PATH}.  Run the fits first or run cell S1."
#     )

# _bundle = _joblib_load(SAVE_PATH)
# print(f"Loaded bundle with {len(_bundle)} keys from {SAVE_PATH}")

# # Rebind every saved name into the global namespace.
# _loaded_names = []
# for _k, _v in _bundle.items():
#     if _v is None:
#         continue
#     globals()[_k] = _v
#     _loaded_names.append(_k)

# print(f"Rebound {len(_loaded_names)} variables into the global namespace")
# print()
# stages = []
# if "ind_fits"     in _loaded_names: stages.append("individual round-1")
# if "ind_fits_v4"  in _loaded_names: stages.append("individual round-2")
# if "p_glob"       in _loaded_names: stages.append("global (Fig. 2)")
# if "refits"       in _loaded_names: stages.append("§4.5 branch-aware refit")
# if "p_glob_A"     in _loaded_names: stages.append("§4.6 Step A")
# if "p_glob_B"     in _loaded_names: stages.append("§4.6 Step B (per-curve C2, a0)")
# if "p_glob_C"     in _loaded_names: stages.append("§4.7 Step C (branch penalty)")
# if "p_glob_D"     in _loaded_names: stages.append("§4.8 Step D (per-curve w_contact)")
# print("Stages available:")
# for s in stages:
#     print(f"   - {s}")

# # Helpful cost summary (printed if available)
# print()
# for _label, _key in [
#     ("global cost   (Fig. 2)", "cost_glob"),
#     ("Step A cost",            "cost_A"),
#     ("Step B cost",            "cost_B"),
#     ("Step C cost (with branch penalty)", "cost_C"),
#     ("Step D cost (with branch penalty)", "cost_D"),
# ]:
#     if _bundle.get(_key, None) is not None:
#         print(f"   {_label:40s}  =  {_bundle[_key]:.4e}")

# # The publication figure cells below can now be executed directly
# # without re-running anything from Sections 1-4.8.
# print("\nReady to make publication figures.")


---

# Publication figure 1 - Per-drive force-model selection from approach + retract

The selector now evaluates candidate force laws separately for each selected drive. For each drive, the measured far-field amplitude `A_far = A0_hat` is treated as the true free-air drive amplitude by tying the simulator drive term to `C3 = A_far / Q` during model selection. This lets the best force model vary with drive amplitude instead of forcing one force law from a single representative curve.


In [ ]:
import os
os.makedirs('output/figs', exist_ok=True)

from codes.spm_force_models_branches import (
    SimOptions, make_force_params, simulate_curve,
    auto_select_force_model, FORCE_MODELS,
)

# Run force-model selection per drive.  Use sel_train by default because those
# are the curves already used by the calibrated workflow; change this to
# good_idx if you want a full-drive model map.
MODEL_SELECTION_IDX = list(sel_train)

# Representative drive used by the publication figure below.
j_pick = MODEL_SELECTION_IDX[len(MODEL_SELECTION_IDX) // 2]
c_app  = curves_app[j_pick]
c_ret  = curves_ret[j_pick]
print(f"Representative figure drive = {drive[j_pick]:.2f} nm")
print(f"Selecting force models for {len(MODEL_SELECTION_IDX)} drives: "
      f"{[round(float(drive[j]), 2) for j in MODEL_SELECTION_IDX]}")


def _c3_from_A_far(A_far_hat, Q):
    # At normalized resonance, free amplitude A_far ~= Q*C3.
    return float(A_far_hat) / max(float(Q), 1e-12)


def _selector_options_for_curve(c):
    A_far_hat = float(c["A0_hat"])
    Q0 = float(p_glob["Q"])
    return SimOptions(
        Q=Q0,
        C3=_c3_from_A_far(A_far_hat, Q0),
        w_contact=float(p_glob["w_contact"]),
        gamma_lr=float(p_glob["gamma_lr"]),
        lambda_lr=float(p_glob["lambda_lr"]),
        gamma_contact=float(p_glob["gamma_contact"]),
        omega_drive_hat=1.0 + float(p_glob["domega"]),
        phase_wrap_mode=360,
        N_cycles=2, max_windows=200, settle_windows=5,
        tol_A=5e-4, tol_phi_deg=0.5, demod_per_ref_cycle=24,
        rtol=1e-4, atol=1e-6, dt0=0.05, dt_max=0.25,
        max_steps_per_window=80000,
    )


# Let the model selector absorb nuisance calibration uncertainty instead of
# forcing every candidate through exactly the same fixed simulator settings.
# C3 is NOT fitted directly here: it is tied to A_far/Q inside
# auto_select_force_model(), so the measured far-field amplitude remains the
# true drive amplitude even when Q is flexible.  All flexible parameters are
# included in each model's AICc penalty.
selector_fit_sim_params = (
    "Q", "w_contact", "gamma_lr", "lambda_lr",
    "gamma_contact", "omega_drive_hat",
)


def _selector_bounds_for_options(opts):
    return {
        "Q": (0.35 * opts.Q, 3.0 * opts.Q),
        "w_contact": (0.20 * opts.w_contact, 5.0 * opts.w_contact),
        "gamma_lr": (0.10 * opts.gamma_lr, 10.0 * opts.gamma_lr),
        "lambda_lr": (0.25 * opts.lambda_lr, 4.0 * opts.lambda_lr),
        "gamma_contact": (0.10 * opts.gamma_contact, 10.0 * opts.gamma_contact),
        "omega_drive_hat": (opts.omega_drive_hat - 0.03,
                            opts.omega_drive_hat + 0.03),
    }


# Subsample the curves for the selector (it iterates many fits).
def sub80(c, n=60):
    n0 = len(c["d_hat"])
    if n0 <= n:
        return c
    idx = np.linspace(0, n0-1, n).astype(int)
    return {**c, "d_hat": c["d_hat"][idx], "A_hat": c["A_hat"][idx],
            "phi_deg": c["phi_deg"][idx]}


# Run one selector per drive, allowing the selected force model to vary with
# drive amplitude.
print("Fitting candidate force models per drive (this can take a few minutes) ...")
import time
t0 = time.time()
force_model_selection_by_drive = {}
selector_options_by_drive = {}

for _ii, j in enumerate(MODEL_SELECTION_IDX, 1):
    ca_i = sub80(curves_app[j])
    cr_i = sub80(curves_ret[j])
    opts_i = _selector_options_for_curve(ca_i)
    A_far_i = float(ca_i["A0_hat"])
    selector_options_by_drive[j] = opts_i

    print(f"\n[{_ii}/{len(MODEL_SELECTION_IDX)}] drive={drive[j]:.2f} nm  "
          f"A_far={A_far_i/conv_L:.3f} nm  "
          f"C3=A_far/Q={_c3_from_A_far(A_far_i, opts_i.Q):.3e}")
    force_model_selection_by_drive[j] = auto_select_force_model(
        d_hat=ca_i["d_hat"], A_exp=ca_i["A_hat"], phi_exp=ca_i["phi_deg"],
        d_ret_hat=cr_i["d_hat"], A_ret=cr_i["A_hat"], phi_ret=cr_i["phi_deg"],
        A0=A_far_i,
        sim_opts=opts_i,
        candidates=("dmt", "lj", "morse", "jkr", "capillary"),
        fit_d_offset=True,
        d_offset_bounds=(-3.0 * conv_L, +3.0 * conv_L),
        fit_sim_params=selector_fit_sim_params,
        sim_param_bounds=_selector_bounds_for_options(opts_i),
        sim_init_overrides={"d_offset_hat": 0.0},
        drive_A_far=A_far_i,
        tie_C3_to_A_far=True,
        max_nfev_per_model=40,
        verbose=True,
    )

best_force_model_by_drive = {
    j: res["best"] for j, res in force_model_selection_by_drive.items()
}
selected_force_params_by_drive = {
    j: next(d["force_params"] for d in res["details"] if d["model"] == res["best"])
    for j, res in force_model_selection_by_drive.items()
}
selected_sim_opts_by_drive = {
    j: next(d.get("sim_opts", selector_options_by_drive[j])
            for d in res["details"] if d["model"] == res["best"])
    for j, res in force_model_selection_by_drive.items()
}

print(f"\nDone in {time.time()-t0:.1f} s")
print("\nBest force model by drive:")
print(f"{'drive(nm)':>10}  {'A_far(nm)':>10}  {'best':>10}  {'ranking'}")
for j in MODEL_SELECTION_IDX:
    res = force_model_selection_by_drive[j]
    A_far_nm = float(curves_app[j]["A0_hat"]) / conv_L
    print(f"{drive[j]:10.2f}  {A_far_nm:10.3f}  {res['best']:>10}  "
          f"{res['ranking']}")

# Backward-compatible aliases for the representative-drive figure below.
sel_result = force_model_selection_by_drive[j_pick]
opts_sel = selector_options_by_drive[j_pick]
c_app = curves_app[j_pick]
c_ret = curves_ret[j_pick]
print(f"\nRepresentative best: drive={drive[j_pick]:.2f} nm -> {sel_result['best']}")


In [ ]:
# -- Build publication figure 1 --
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

# Color palette for force models (colorblind-friendly)
MODEL_COLORS = {
    "dmt":       "#1b9e77",
    "lj":        "#d95f02",
    "morse":     "#7570b3",
    "jkr":       "#e7298a",
    "capillary": "#66a61e",
}
MODEL_LABEL = {
    "dmt":       "DMT",
    "lj":        "Lennard-Jones",
    "morse":     "Morse",
    "jkr":       "JKR (adhesive)",
    "capillary": "Capillary",
}

# Re-simulate each fit on the FULL curve for plotting
sim_per_model = {}
for det in sel_result["details"]:
    name = det["model"]
    p_fit = det["force_params"]
    opts_fit = det.get("sim_opts", opts_sel)
    try:
        out = simulate_curve(c_app["d_hat"], name, p_fit, opts_fit)
        sim_per_model[name] = (out["A"], out["phi"])
    except Exception as e:
        print(f"  failed to re-sim {name}: {e}")

# Compute hysteresis between approach and retract on a common d grid
order_r = np.argsort(c_ret["d_hat"])
A_ret_on_app = np.interp(c_app["d_hat"], c_ret["d_hat"][order_r], c_ret["A_hat"][order_r])
hyst = A_app_minus_ret = c_app["A_hat"] - A_ret_on_app

plt.rcParams.update({"font.size": 9, "axes.linewidth": 0.8,
                     "xtick.major.width": 0.7, "ytick.major.width": 0.7})

fig = plt.figure(figsize=(7.5, 5.0), dpi=150)
gs = fig.add_gridspec(2, 3, height_ratios=[1, 1], width_ratios=[1, 1, 1],
                      hspace=0.42, wspace=0.42)
axA  = fig.add_subplot(gs[0, 0:2])    # approach + retract amplitude w/ model fits
axP  = fig.add_subplot(gs[1, 0:2])    # phase comparison
axH  = fig.add_subplot(gs[0, 2])      # hysteresis (A_app - A_ret)
axS  = fig.add_subplot(gs[1, 2])      # model scores

# (a) Amplitude overlay
d_nm = c_app["d_hat"]/conv_L
dr_nm = c_ret["d_hat"]/conv_L
axA.plot(d_nm, c_app["A_hat"]/conv_L, 'ko', ms=2.5, mfc='white',
         mew=0.6, label='exp (approach)')
axA.plot(dr_nm, c_ret["A_hat"]/conv_L, 'k^', ms=2.5, mfc='white',
         mew=0.6, label='exp (retract)')
for name, (A_sim, _) in sim_per_model.items():
    axA.plot(d_nm, A_sim/conv_L, '-', color=MODEL_COLORS[name],
             lw=1.2, label=MODEL_LABEL[name], alpha=0.85)
axA.set_ylabel("Amplitude (nm)")
axA.set_xlabel("Tip-sample distance (nm)")
axA.legend(fontsize=7, frameon=False, ncol=2, loc='lower right')
axA.text(-0.10, 1.03, "(a)", transform=axA.transAxes, fontsize=11,
         fontweight='bold')
axA.set_title(f"Drive = {drive[j_pick]:.2f} nm; best = {sel_result['best']}",
              fontsize=9, loc='right')

# (b) Phase comparison
from codes.spm_fd_rk45_extended_calibration import (
    transform_phase_deg, estimate_phase_offset_far_field,
)
axP.plot(d_nm, c_app["phi_deg"], 'ko', ms=2.5, mfc='white',
         mew=0.6, label='exp (approach)')
axP.plot(dr_nm, c_ret["phi_deg"], 'k^', ms=2.5, mfc='white',
         mew=0.6, label='exp (retract)')
for name, (_, phi_sim) in sim_per_model.items():
    phi_use = transform_phase_deg(phi_sim, mode=PHASE_MODE)
    offset = estimate_phase_offset_far_field(
        phi_use, c_app["phi_deg"], c_app["d_hat"],
        frac=0.15, period_deg=360.0)
    axP.plot(d_nm, phi_use + offset, '-', color=MODEL_COLORS[name],
             lw=1.2, alpha=0.85)
axP.set_xlabel("Tip-sample distance (nm)")
axP.set_ylabel("Phase (deg)")
axP.text(-0.10, 1.03, "(b)", transform=axP.transAxes, fontsize=11,
         fontweight='bold')

# (c) Hysteresis
axH.plot(d_nm, hyst/conv_L, 'k.-', ms=2, lw=0.7)
axH.axhline(0, color='gray', lw=0.5)
axH.set_xlabel("Distance (nm)")
axH.set_ylabel("A_app - A_ret (nm)")
axH.text(-0.20, 1.03, "(c)", transform=axH.transAxes, fontsize=11,
         fontweight='bold')
axH.set_title("Hysteresis", fontsize=9)

# (d) Model scores (lower AICc-adjusted = better)
names_ord = sel_result["ranking"]
aicc = [next(d["aicc_adjusted"] for d in sel_result["details"]
             if d["model"] == n) for n in names_ord]
colors_ord = [MODEL_COLORS[n] for n in names_ord]
labels_ord = [MODEL_LABEL[n] for n in names_ord]
x = np.arange(len(names_ord))
axS.barh(x, np.array(aicc) - min(aicc), color=colors_ord, height=0.7)
axS.set_yticks(x); axS.set_yticklabels(labels_ord, fontsize=8)
axS.invert_yaxis()
axS.set_xlabel(r"$\Delta$AICc"); axS.set_title("Model ranking", fontsize=9)
axS.text(-0.30, 1.03, "(d)", transform=axS.transAxes, fontsize=11,
         fontweight='bold')

# Highlight best model
best = sel_result["best"]
for ax in (axA, axP):
    # rebold the best model's line
    for ln in ax.get_lines():
        if ln.get_label() == MODEL_LABEL[best]:
            ln.set_linewidth(2.2); ln.set_alpha(1.0)
axA.text(0.02, 0.05, f"Best: {MODEL_LABEL[best]}", transform=axA.transAxes,
         fontsize=9, color=MODEL_COLORS[best], fontweight='bold')

fig.savefig("output/figs/fig1_force_model_selection.pdf", bbox_inches="tight")
fig.savefig("output/figs/fig1_force_model_selection.png", bbox_inches="tight",
            dpi=300)
print("Saved output/figs/fig1_force_model_selection.{pdf,png}")
plt.show()


---

# Publication figure 2 — Individual-to-global fitting workflow

Three representative training drives at three stages of the fitting
pipeline:

1.  **Stage 1 (left)**: per-curve independent fits with all 7 parameters
    free.  Visually OK overlays but each curve picks its own (Q, C2, a0,
    d_off) basin → physically inconsistent.

2.  **Stage 2 (middle)**: shared-physics global fit.  C1, C2, a0, Q, and
    damping are tied across all curves.  Per-curve C3, d_off, phi_off
    absorb instrument-specific corrections.

3.  **Stage 3 (right, optional)**: round-2 refit with shared physics
    locked.  Verifies that the shared physics fits every curve, not just
    on average.

Bottom row: per-curve `Q` and `C2` from stage 1 vs the single shared
values from stage 2.  Tight scatter → the data agrees on one set of
physics.

In [ ]:
# -- Build publication figure 2 --
import matplotlib.pyplot as plt

# Pick 3 representative drives: high (clean repulsive), mid (just below
# the lower fold of the tapping-mode bistability, so cleanly attractive),
# and low (clean attractive).  The original [0, 4, 7] mid choice landed
# inside the bistable strip (drive = 11.37 nm), where simulator and
# experiment occupy different branches — see Section 4.6-4.8 for the
# diagnosis.  We swap to k = 5 (drive = 8.35 nm) which is outside the
# bistable region.
MID_K_DRIVE = 5
pick_idx = [0, MID_K_DRIVE, len(sel_train) - 1]
pick_drives = [sel_train[i] for i in pick_idx]

# Recompute per-curve simulated traces at three stages
omega_base = 1.0 + p_glob["domega"]

def sim_with(params_dict, curve, drv_local):
    A_sim, phi_sim, *_ = simulate_curve_rk45_extended_python(
        curve["d_hat"],
        C1=params_dict["C1"], C2=params_dict["C2"], a0=params_dict["a0"],
        Q=params_dict["Q"], C3=params_dict["C3"],
        omega_drive_hat=params_dict.get("omega_d", 1.0),
        omega_ref_hat=params_dict.get("omega_d", 1.0),
        d_offset_hat=params_dict["d_off"],
        w_contact=params_dict["w_contact"], gamma_lr=params_dict["gamma_lr"],
        lambda_lr=params_dict["lambda_lr"], gamma_contact=params_dict["gamma_contact"],
        **rk45_plot_options,
    )
    return A_sim, phi_sim

plt.rcParams.update({"font.size": 9, "axes.linewidth": 0.8})

fig = plt.figure(figsize=(8.5, 6.2), dpi=150)
gs = fig.add_gridspec(3, 3, height_ratios=[1, 1, 0.85], hspace=0.55, wspace=0.35)

# Top row: 3 curves x 3 stages would be too busy. Instead: 3 columns = 3 drives
# Row 1: amplitude with individual (red) + global (blue) overlays
# Row 2: phase with offsets removed (individual red, global blue)
# Row 3: parameter consistency: Q and C2 per curve

stage_labels = ['Individual fit', 'Global fit (shared physics)', 'Round-2 refit']
stage_colors = ['#d95f02', '#1b9e77', '#7570b3']

for col, j_global in enumerate(pick_drives):
    c = curves_app_sub[j_global]
    k_train = sel_train.index(j_global)
    f1 = ind_fits[k_train]
    f3 = ind_fits_v4[k_train]
    d_nm = c["d_hat"]/conv_L

    # Amplitude row
    axA = fig.add_subplot(gs[0, col])
    axA.plot(d_nm, c["A_hat"]/conv_L, 'k.', ms=2.5, mfc='white', mew=0.5,
             alpha=0.7, label='exp')
    # Stage 1: individual
    p_ind = dict(C1=f1["C1"], C2=f1["C2"], a0=f1["a0"], Q=f1["Q"], C3=f1["C3"],
                 d_off=f1["d_offset_hat"], w_contact=fixed_extra["w_contact"],
                 gamma_lr=fixed_extra["gamma_lr"],
                 lambda_lr=fixed_extra["lambda_lr"],
                 gamma_contact=fixed_extra["gamma_contact"], omega_d=1.0)
    A1, _ = sim_with(p_ind, c, drive[j_global])
    axA.plot(d_nm, A1/conv_L, '-', color=stage_colors[0], lw=1.2,
             label='individual', alpha=0.95)
    # Stage 2: global (round-2 C3, d_off; shared physics)
    p_gl = dict(C1=p_glob["C1"], C2=p_glob["C2"], a0=p_glob["a0"],
                Q=p_glob["Q"], C3=f3["C3"],
                d_off=f3["d_off"], w_contact=p_glob["w_contact"],
                gamma_lr=p_glob["gamma_lr"], lambda_lr=p_glob["lambda_lr"],
                gamma_contact=p_glob["gamma_contact"], omega_d=omega_base)
    A2, _ = sim_with(p_gl, c, drive[j_global])
    axA.plot(d_nm, A2/conv_L, '-', color=stage_colors[1], lw=1.2,
             label='global', alpha=0.95)
    axA.set_title(f"drive = {drive[j_global]:.2f} nm", fontsize=9)
    axA.set_xlabel("Height (nm)", fontsize=8)
    if col == 0:
        axA.set_ylabel("Amplitude (nm)", fontsize=8)
        axA.legend(fontsize=7, frameon=False)
        axA.text(-0.22, 1.07, "(a)", transform=axA.transAxes, fontsize=11,
                 fontweight='bold')
    axA.tick_params(labelsize=7)

    # Phase row (offset removed)
    axP = fig.add_subplot(gs[1, col])
    n_far = max(3, int(0.15 * len(c["d_hat"])))
    far_idx = np.argsort(c["d_hat"])[::-1][:n_far]

    _, phi1 = sim_with(p_ind, c, drive[j_global])
    phi1u = transform_phase_deg(phi1, mode=PHASE_MODE)
    o1 = estimate_phase_offset_far_field(phi1u, c["phi_deg"], c["d_hat"],
                                          frac=0.15, period_deg=360.0)
    phi1u = phi1u + o1 + f1.get("phi_offset_deg", 0.0)
    phi1u_far = float(np.mean(phi1u[far_idx]))

    _, phi2 = sim_with(p_gl, c, drive[j_global])
    phi2u = transform_phase_deg(phi2, mode=PHASE_MODE)
    o2 = estimate_phase_offset_far_field(phi2u, c["phi_deg"], c["d_hat"],
                                          frac=0.15, period_deg=360.0)
    phi2u = phi2u + o2 + f3["phi_off"]
    phi2u_far = float(np.mean(phi2u[far_idx]))

    phi_exp_far = float(np.mean(c["phi_deg"][far_idx]))

    axP.plot(d_nm, c["phi_deg"] - phi_exp_far, 'k.', ms=2.5, mfc='white',
             mew=0.5, alpha=0.7)
    axP.plot(d_nm, phi1u - phi1u_far, '-', color=stage_colors[0], lw=1.2,
             alpha=0.95)
    axP.plot(d_nm, phi2u - phi2u_far, '-', color=stage_colors[1], lw=1.2,
             alpha=0.95)
    axP.axhline(0, color='gray', lw=0.4, ls='--')
    axP.set_xlabel("Height (nm)", fontsize=8)
    if col == 0:
        axP.set_ylabel(r"$\phi - \phi_{far}$ (deg)", fontsize=8)
        axP.text(-0.22, 1.07, "(b)", transform=axP.transAxes, fontsize=11,
                 fontweight='bold')
    axP.tick_params(labelsize=7)

# Bottom row: parameter consistency (Q and C2)
axQ = fig.add_subplot(gs[2, 0])
Q_ind = np.array([f["Q"] for f in ind_fits])
drives_train = np.array([f["drive_nm"] for f in ind_fits])
axQ.plot(drives_train, Q_ind, 'o', color=stage_colors[0], ms=5, mec='k',
         mew=0.4, label='individual')
axQ.axhline(p_glob["Q"], color=stage_colors[1], lw=1.5, ls='--',
            label=f"global Q = {p_glob['Q']:.1f}")
axQ.set_xlabel("Drive amplitude (nm)", fontsize=8)
axQ.set_ylabel("Q  (quality factor)", fontsize=8)
axQ.legend(fontsize=7, frameon=False)
axQ.text(-0.22, 1.05, "(c)", transform=axQ.transAxes, fontsize=11,
         fontweight='bold')
axQ.tick_params(labelsize=7)

axC2 = fig.add_subplot(gs[2, 1])
C2_ind = np.array([f["C2"] for f in ind_fits])
axC2.plot(drives_train, C2_ind, 's', color=stage_colors[0], ms=5, mec='k',
          mew=0.4, label='individual')
axC2.axhline(p_glob["C2"], color=stage_colors[1], lw=1.5, ls='--',
             label=f"global C2 = {p_glob['C2']:.2f}")
axC2.set_xlabel("Drive amplitude (nm)", fontsize=8)
axC2.set_ylabel(r"$C_2$  (Hertz stiffness, hat)", fontsize=8)
axC2.set_yscale('log')
axC2.legend(fontsize=7, frameon=False)
axC2.text(-0.22, 1.05, "(d)", transform=axC2.transAxes, fontsize=11,
          fontweight='bold')
axC2.tick_params(labelsize=7)

# Cost comparison: individual vs round-2
axE = fig.add_subplot(gs[2, 2])
cost_ind = np.array([f["cost"] for f in ind_fits])
cost_r2  = np.array([r["cost"] for r in ind_fits_v4])
axE.plot(drives_train, cost_ind, 'o-', color=stage_colors[0], ms=4, lw=0.8,
         label='individual')
axE.plot(drives_train, cost_r2,  's-', color=stage_colors[2], ms=4, lw=0.8,
         label='round-2 (shared)')
axE.set_xlabel("Drive amplitude (nm)", fontsize=8)
axE.set_ylabel("Fit cost", fontsize=8)
axE.set_yscale('log')
axE.legend(fontsize=7, frameon=False)
axE.text(-0.22, 1.05, "(e)", transform=axE.transAxes, fontsize=11,
         fontweight='bold')
axE.tick_params(labelsize=7)

fig.suptitle("Individual-to-global fitting workflow", y=1.0, fontsize=11)
plt.tight_layout()
fig.savefig("output/figs/fig2_workflow.pdf", bbox_inches="tight")
fig.savefig("output/figs/fig2_workflow.png", bbox_inches="tight", dpi=300)
print("Saved output/figs/fig2_workflow.{pdf,png}")
plt.show()


## Nature-style manuscript figure export

This final cell exports the manuscript figure package with the shared Nature-style layout contract: 183 mm print width, white background, lowercase panel labels, and high-DPI PNG/TIFF/PDF outputs under `output/nature_figures`.

In [ ]:
from pathlib import Path
import importlib
import codes.nature_style_outputs as nature_style_outputs

importlib.reload(nature_style_outputs)
project_root = Path.cwd().resolve()
nature_figures = nature_style_outputs.build_fd_calibration_figures(project_root)
for figure_name, outputs in nature_figures.items():
    print(figure_name)
    for fmt, path in outputs.items():
        print(f"  {fmt}: {path}")
